# ☸️ Kubeflow Pipelines — ML Workflows on Kubernetes (Optional)

> **What you'll learn:** the Kubernetes ideas every ML engineer needs, what the Kubeflow ecosystem (Pipelines, Katib, KServe, Trainer) does, and how to build **KFP v2** pipelines: lightweight Python components, typed artifacts, parameters, control flow (`dsl.If`, `ParallelFor`, `ExitHandler`), caching, and the compiled IR YAML that Kubeflow and Vertex AI run. Everything executes for real on your laptop with **KFP local execution** — no cluster required.
>
> **Optional notebook:** skim it if your target roles don't mention Kubeflow, Vertex AI Pipelines or Kubernetes-based ML platforms.

| | |
|---|---|
| **Difficulty** | 🟡 Intermediate → 🔴 Advanced (platform concepts) |
| **Time** | ~3 hours to read and run, +1.5 hours for exercises and the project |
| **Prerequisites** | [Airflow](04_Airflow.ipynb) (orchestration, DAGs, idempotency) · [Model Monitoring and Drift](05_Model_Monitoring_and_Drift.ipynb) (why models are retrained) |
| **Tested with** | Python 3.12 · kfp 2.17 · scikit-learn 1.9 · pandas 3.0 — in its **own** virtual environment; local runs need no Kubernetes or Docker |
| **Interview relevance** | ⭐⭐ Medium (⭐⭐⭐ for ML-platform roles) — containers/pods/resources, artifacts vs parameters, pipeline caching, Katib/KServe, "KFP or Airflow?" |

## 🤔 What Is Kubeflow Pipelines?

Picture a **factory with shipping containers**. Every workstation (download data, train, evaluate) comes packed in its own sealed container with exactly the tools it needs. A conveyor belt moves labelled boxes (datasets, models, metrics) between workstations, and a logbook records every box that went through.

- **Kubernetes** (K8s) is the factory floor: it runs containers on a cluster of machines.
- **Kubeflow** is a collection of ML tools that run on Kubernetes.
- **Kubeflow Pipelines (KFP)** is the conveyor belt and logbook: you write a pipeline in Python, KFP **compiles** it into a portable spec, and a backend runs **each step as its own container**, passing typed **artifacts** between steps and tracking lineage.

```
  Python (@dsl.component, @dsl.pipeline) ──compile──► pipeline spec (IR YAML) ──► Kubeflow Pipelines on K8s
                                                                               └─► Vertex AI Pipelines (managed)
                                                  └── kfp.local ──► runs the same pipeline on your laptop
```

Words used below: a **component** is one step (a function or container); a **task** is one use of a component inside a pipeline; a **parameter** is a small value (int, str, dict); an **artifact** is a file-like output with metadata (dataset, model, metrics).

## 🎯 Why It Matters

- **ML platforms at many companies run on Kubernetes.** Training jobs, batch inference and pipelines are pods; knowing requests/limits, images and namespaces is expected of ML engineers who deploy their own work.
- **Vertex AI Pipelines runs the KFP spec.** If a company uses Google Cloud for ML, "write a KFP pipeline" is often literally the job.
- **Artifacts and lineage** — "which data and code produced this model?" — are core MLOps interview topics, and KFP makes them first-class.
- **In interviews** you'll hear: "What's the difference between a parameter and an artifact?", "How does pipeline caching decide to skip a step?", "Requests vs limits?", "How would you tune hyperparameters on Kubernetes?", "KFP or Airflow?" This notebook answers them with pipelines that actually run.

## ✅ By the End You Can

- [ ] Explain containers, images, pods, namespaces and CPU/memory requests vs limits
- [ ] Describe the Kubeflow ecosystem (Pipelines, Katib, KServe, Trainer) and its managed equivalents
- [ ] Write lightweight Python components with typed parameters and artifacts, and run them locally
- [ ] Build pipelines with `dsl.If`/`Elif`/`Else`, `ParallelFor`, `ExitHandler` and caching, and read the compiled IR YAML
- [ ] Write a valid Katib `Experiment` and KServe `InferenceService`, and explain every field
- [ ] Choose between KFP and Airflow for a given ML workflow

## 📋 Table of Contents

1. [Kubernetes Concepts an ML Engineer Needs](#1.-Kubernetes-Concepts-an-ML-Engineer-Needs-🟢)
2. [The Kubeflow Ecosystem](#2.-The-Kubeflow-Ecosystem-🟢)
3. [Lightweight Python Components](#3.-Lightweight-Python-Components-🟢)
4. [Parameters vs Artifacts](#4.-Parameters-vs-Artifacts-🟡)
5. [Building and Running a Pipeline Locally](#5.-Building-and-Running-a-Pipeline-Locally-🟡)
6. [Control Flow: If, ParallelFor, ExitHandler](#6.-Control-Flow:-If,-ParallelFor,-ExitHandler-🔴)
7. [Caching](#7.-Caching-🟡)
8. [Compiling to IR YAML and Reading It](#8.-Compiling-to-IR-YAML-and-Reading-It-🟡)
9. [Katib: Hyperparameter Tuning Experiments](#9.-Katib:-Hyperparameter-Tuning-Experiments-🟡)
10. [KServe: Serving with an InferenceService](#10.-KServe:-Serving-with-an-InferenceService-🟡)
11. [Running on a Cluster, and KFP vs Airflow](#11.-Running-on-a-Cluster,-and-KFP-vs-Airflow-🟢)
- [🔧 Build It From Scratch](#🔧-Build-It-From-Scratch) · [⚠️ Common Pitfalls](#⚠️-Common-Pitfalls) · [🏋️ Practice Exercises](#🏋️-Practice-Exercises) · [🚀 Mini Project](#🚀-Mini-Project:-Train-→-Evaluate-→-Conditional-Register) · [🎤 Interview Q&A](#🎤-Interview-Q&A) · [🧪 Quick Quiz](#🧪-Quick-Quiz) · [📚 Resources](#📚-Resources) · [📝 Summary](#📝-Summary-Cheat-Sheet)

## ⚙️ Setup

**Use a separate virtual environment** (the KFP SDK pins protobuf and Kubernetes client versions that often clash with other ML stacks):

```bash
uv venv .venvs/kfp --python 3.12
uv pip install --python .venvs/kfp "kfp==2.17.*" scikit-learn pandas pyyaml ipykernel
```

**How pipelines run here.** `kfp.local.init(runner=kfp.local.SubprocessRunner(use_venv=False))` runs every task as a **separate Python subprocess** using this environment, stores outputs under a local *pipeline root* (`_outputs/kfp/pipeline_root`, shown as `$PIPELINE_ROOT`), and supports conditions, loops, exit handlers and caching. Two honest differences from a cluster:

1. On a cluster each task runs inside its `base_image` container, which starts by `pip install`-ing `kfp` (and your `packages_to_install`). Locally, this environment already has everything, so our components pass `install_kfp_package=False` and keep imports to packages installed here.
2. Kubernetes-only settings (CPU/memory requests, GPUs, retries on the pod) are **compiled into the spec** but do nothing locally.

`kfp.local.DockerRunner()` runs each task in its real container image — its cell below only runs if Docker is available. Submitting to a real cluster needs a KFP endpoint in the `KFP_HOST` environment variable; otherwise that cell prints a skip message.

In [1]:
# %pip install "kfp==2.17.*" scikit-learn pandas pyyaml

import builtins
import contextlib
import importlib.metadata as md
import importlib.util
import io
import json
import logging
import math
import os
import re
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path
from typing import List, NamedTuple

import kfp
import yaml
from kfp import compiler, dsl, local
from kfp.dsl import ClassificationMetrics, Dataset, Input, Metrics, Model, Output

OUTPUT_DIR = Path("_outputs")
KFP_DIR = (OUTPUT_DIR / "kfp").resolve()
shutil.rmtree(KFP_DIR, ignore_errors=True)            # fresh outputs and cache every run → reproducible
PIPELINE_ROOT = KFP_DIR / "pipeline_root"
CACHE_ROOT = KFP_DIR / "cache"
SPECS = KFP_DIR / "specs"                             # compiled YAML files
LOG_FILE = KFP_DIR / "local_runs.log"
for folder in (PIPELINE_ROOT, CACHE_ROOT, SPECS):
    folder.mkdir(parents=True)

def init_local(enable_caching=True):
    """(Re)configure local execution: every task is a subprocess of this environment; outputs go to PIPELINE_ROOT."""
    local.init(runner=local.SubprocessRunner(use_venv=False), pipeline_root=str(PIPELINE_ROOT),
               enable_caching=enable_caching, cache_root=str(CACHE_ROOT), raise_on_error=False)


init_local()

# Component settings for local runs (see the note above); on a cluster you'd drop install_kfp_package=False
LOCAL = dict(base_image="python:3.12", install_kfp_package=False)

docker = shutil.which("docker")
DOCKER_DAEMON = bool(docker) and subprocess.run([docker, "info"], capture_output=True).returncode == 0
DOCKER_SDK = importlib.util.find_spec("docker") is not None    # DockerRunner also needs the `docker` Python package
KFP_HOST = os.environ.get("KFP_HOST")

print(f"Python {sys.version.split()[0]} | kfp {kfp.__version__} | scikit-learn {md.version('scikit-learn')} | "
      f"pandas {md.version('pandas')} | pyyaml {md.version('pyyaml')}")
print("Docker daemon running:", DOCKER_DAEMON, "| docker Python package:", DOCKER_SDK,
      "| KFP cluster endpoint (KFP_HOST):", KFP_HOST or "not set")


def rel(text):
    """Hide machine-specific absolute paths in printed output."""
    text = str(text).replace(str(PIPELINE_ROOT), "$PIPELINE_ROOT").replace(str(KFP_DIR), "$KFP_DIR")
    return re.sub(r"/(?:private/)?(?:var/folders|tmp)/\S+", "<tmp>", text)


def run_local(fn, show_local_notes=True, **kwargs):
    """Run a component or pipeline locally; capture KFP's own logs to LOG_FILE and return (task, seconds, log_text)."""
    buffer = io.StringIO()
    handler = logging.StreamHandler(buffer)
    root_logger = logging.getLogger()
    old_level = root_logger.level
    root_logger.addHandler(handler)
    root_logger.setLevel(logging.INFO)                # KFP reports cache hits through the root logger
    start = time.perf_counter()
    original_print = builtins.print                   # KFP indents prints during a run and can leave that on after a failure
    try:
        with warnings.catch_warnings(record=True) as caught, contextlib.redirect_stdout(buffer), contextlib.redirect_stderr(buffer):
            warnings.simplefilter("always")
            task = fn(**kwargs)
    finally:
        builtins.print = original_print
        root_logger.removeHandler(handler)
        root_logger.setLevel(old_level)
    seconds = time.perf_counter() - start
    notes = sorted({str(w.message) for w in caught})
    if show_local_notes:
        for note in notes:                            # e.g. "settings ['cpu_request'] have no effect in the current local runner"
            print("ℹ️ local runner:", note)
    text = re.sub(r"\x1b\[[0-9;]*m", "", buffer.getvalue())
    with LOG_FILE.open("a") as f:
        f.write(text)
    return task, seconds, text


def task_statuses(log_text):
    """[(task, status)] in the order tasks finished, parsed from KFP's local log."""
    return re.findall(r"Task '([\w-]+)' finished with status (\w+)", log_text)


def latest_run_dir(pipeline_name):
    runs = sorted(PIPELINE_ROOT.glob(f"{pipeline_name}-*"), key=lambda p: p.stat().st_mtime)
    return runs[-1]


def check(name, got, expected, hint=""):
    """✅ if correct, ⏳ if not attempted yet (None), ❌ AssertionError with a hint otherwise."""
    if got is None or got is ...:
        print(f"⏳ {name}: not attempted yet — replace None with your answer.")
        return
    if isinstance(expected, float) and isinstance(got, (int, float)):
        ok = math.isclose(got, expected, rel_tol=1e-6, abs_tol=1e-9)
    else:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    assert ok, f"❌ {name}: not quite (got {got!r}). {hint}"
    print(f"✅ {name}: correct!")

Python 3.12.11 | kfp 2.17.0 | scikit-learn 1.9.1 | pandas 3.0.5 | pyyaml 6.0.3
Docker daemon running: True | docker Python package: True | KFP cluster endpoint (KFP_HOST): not set


## 1. Kubernetes Concepts an ML Engineer Needs 🟢

| Term | Plain English | ML example |
|---|---|---|
| **Container** | a process packaged with its own filesystem (code + libraries) | a training script with PyTorch 2.x |
| **Image** | the read-only template a container starts from, stored in a registry | `python:3.12`, `ghcr.io/team/trainer:1.4.2` |
| **Pod** | the smallest thing K8s runs: one or more containers sharing network/storage | one pipeline step, one serving replica |
| **Node** | a machine (VM) in the cluster | an 8-GPU node pool for training |
| **Namespace** | a named partition for access control and quotas | `team-recsys`, `kubeflow-user-alice` |
| **Requests** | resources the scheduler **reserves** — used to decide *where* a pod fits | `cpu: 500m` (half a core), `memory: 2Gi` |
| **Limits** | the **ceiling** at run time: CPU above it is throttled; memory above it gets the container **OOMKilled** | `memory: 4Gi` |
| **Job / Deployment** | run-to-completion workload / long-running replicas | a training job / a model server |
| **PersistentVolumeClaim** | a request for durable disk | a shared dataset cache |
| **ServiceAccount** | the identity a pod uses to call cloud APIs | read access to `gs://ml-data` |

CPU is measured in **cores** (`1`, `0.5`, or millicores `500m`); memory in bytes with binary suffixes (`Mi` = 2²⁰, `Gi` = 2³⁰) or decimal ones (`M` = 10⁶, `G` = 10⁹).

In [2]:
POD_SPEC = yaml.safe_load("""
apiVersion: v1
kind: Pod
metadata:
  name: train-churn-model
  namespace: team-recsys
spec:
  restartPolicy: Never
  containers:
    - name: trainer
      image: ghcr.io/example/churn-trainer:1.4.2
      command: ["python", "train.py", "--epochs", "10"]
      resources:
        requests: {cpu: "1500m", memory: "3Gi"}
        limits:   {cpu: "2",     memory: "4Gi", nvidia.com/gpu: 1}
""")

SUFFIX = {"Ki": 2**10, "Mi": 2**20, "Gi": 2**30, "Ti": 2**40, "k": 10**3, "M": 10**6, "G": 10**9, "T": 10**12}


def cpu_cores(quantity):
    quantity = str(quantity)
    return int(quantity[:-1]) / 1000 if quantity.endswith("m") else float(quantity)


def memory_bytes(quantity):
    match = re.fullmatch(r"(\d+(?:\.\d+)?)([A-Za-z]*)", str(quantity))
    number, suffix = match.groups()
    return int(float(number) * SUFFIX.get(suffix, 1))


resources = POD_SPEC["spec"]["containers"][0]["resources"]
print(f"requests: {cpu_cores(resources['requests']['cpu'])} cores, {memory_bytes(resources['requests']['memory']) / 2**30:.1f} GiB")
print(f"limits  : {cpu_cores(resources['limits']['cpu'])} cores, {memory_bytes(resources['limits']['memory']) / 2**30:.1f} GiB, "
      f"{resources['limits']['nvidia.com/gpu']} GPU")
print("'4Gi' vs '4G' bytes:", memory_bytes("4Gi"), "vs", memory_bytes("4G"), "— Gi is 7.4% bigger")

requests: 1.5 cores, 3.0 GiB
limits  : 2.0 cores, 4.0 GiB, 1 GPU
'4Gi' vs '4G' bytes: 4294967296 vs 4000000000 — Gi is 7.4% bigger


### ✍️ Your Turn

A node has **4 CPU cores**, of which **0.5 cores** are reserved for system processes. How many pods requesting `cpu: 500m` can the scheduler place on it (CPU only)? Store the integer in `pods_that_fit` — use `cpu_cores()`.

In [3]:
pods_that_fit = None  # TODO: your code here
check("pods_that_fit", pods_that_fit, 7, hint="int((4 - 0.5) // cpu_cores('500m'))")

⏳ pods_that_fit: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
pods_that_fit = int((4 - 0.5) // cpu_cores("500m"))
check("pods_that_fit", pods_that_fit, 7)
```

Scheduling uses **requests**, not actual usage — which is why over-requesting wastes clusters and under-requesting causes noisy-neighbour slowdowns.
</details>

> 💡 **Interview angle:** "Requests vs limits?" — requests are reserved and drive scheduling; limits cap usage. Exceeding a CPU limit throttles the container; exceeding a memory limit gets it **OOMKilled** (exit code 137). For training jobs, set memory request ≈ limit to avoid surprise evictions.

## 2. The Kubeflow Ecosystem 🟢

| Component | What it does | Managed / similar |
|---|---|---|
| **Kubeflow Pipelines (KFP)** | author, compile, run and track ML pipelines; artifacts, lineage, caching | **Vertex AI Pipelines** runs KFP's compiled spec directly |
| **Katib** | hyperparameter tuning, early stopping, neural architecture search as K8s `Experiment`s | Vertex AI Vizier, SageMaker Automatic Model Tuning |
| **KServe** | model serving: `InferenceService` with autoscaling (incl. scale-to-zero), canary rollouts, standard inference protocols | Vertex AI Endpoints, SageMaker endpoints |
| **Trainer** (formerly Training Operator) | distributed training jobs (PyTorch, JAX, …) on K8s | Vertex AI Training, SageMaker Training |
| **Notebooks** | Jupyter servers as pods, per-user namespaces | Vertex AI Workbench, SageMaker Studio |
| **Model Registry** | metadata about model versions and where they're stored | MLflow Model Registry, Vertex Model Registry |

You can install just **KFP standalone** on a cluster, or the whole Kubeflow platform. The Python SDK (`kfp`) is the same either way.

In [4]:
print("kfp.dsl       → authoring:", ", ".join(n for n in ("component", "pipeline", "If", "Elif", "Else", "ParallelFor", "ExitHandler", "Collected") if hasattr(dsl, n)))
print("kfp.dsl types → artifacts:", ", ".join(n for n in ("Artifact", "Dataset", "Model", "Metrics", "ClassificationMetrics", "HTML", "Markdown") if hasattr(dsl, n)))
print("kfp.compiler  → Compiler().compile(pipeline, 'pipeline.yaml')")
print("kfp.local     → runners:", ", ".join(n for n in ("SubprocessRunner", "DockerRunner") if hasattr(local, n)))
print("kfp.Client    → submit to a KFP backend:", kfp.Client.__name__)

kfp.dsl       → authoring: component, pipeline, If, Elif, Else, ParallelFor, ExitHandler, Collected
kfp.dsl types → artifacts: Artifact, Dataset, Model, Metrics, ClassificationMetrics, HTML, Markdown
kfp.compiler  → Compiler().compile(pipeline, 'pipeline.yaml')
kfp.local     → runners: SubprocessRunner, DockerRunner
kfp.Client    → submit to a KFP backend: Client


> 💡 **Interview angle:** "What is Kubeflow?" — not one tool but a set of Kubernetes-native ML components: Pipelines for workflows, Katib for tuning, KServe for serving, Trainer for distributed training. On Google Cloud, Vertex AI Pipelines executes the same KFP pipeline spec without you running Kubeflow.

## 3. Lightweight Python Components 🟢

A **lightweight Python component** is a normal function turned into a pipeline step by `@dsl.component`:

- Everything the function needs must be **inside the function**: imports, helpers, constants. KFP ships only the function's source code into the container.
- Arguments and return values need **type hints** — KFP uses them to build the component's interface.
- `base_image` picks the container image; `packages_to_install` pip-installs extra packages when the container starts (for production, bake them into a custom image instead — faster and reproducible).

Calling a component after `local.init()` **runs it right away** as a subprocess and returns a task whose `.output` holds the result.

In [5]:
@dsl.component(**LOCAL)
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


print("inputs :", {name: spec.type for name, spec in add.component_spec.inputs.items()})
print("outputs:", {name: spec.type for name, spec in add.component_spec.outputs.items()})

task, seconds, log_text = run_local(add, a=2, b=40)
print(f"add(2, 40) → {task.output}   (ran as a separate process in {seconds:.1f} s)")

inputs : {'a': 'Integer', 'b': 'Integer'}
outputs: {'Output': 'Integer'}
add(2, 40) → 42   (ran as a separate process in 0.1 s)


What does `packages_to_install` actually do? Compile a component that uses it and read the container command KFP generates:

In [6]:
@dsl.component(base_image="python:3.12-slim", packages_to_install=["scikit-learn==1.9.1", "pandas>=2.2"])
def train_on_cluster(n_estimators: int) -> float:
    from sklearn.datasets import load_iris
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    X, y = load_iris(return_X_y=True)
    return float(cross_val_score(RandomForestClassifier(n_estimators=n_estimators, random_state=0), X, y).mean())


compiler.Compiler().compile(train_on_cluster, str(SPECS / "train_on_cluster.yaml"))
container = yaml.safe_load((SPECS / "train_on_cluster.yaml").read_text())["deploymentSpec"]["executors"]["exec-train-on-cluster"]["container"]
print("image:", container["image"])
pip_lines = [line.strip() for line in container["command"][2].splitlines() if "pip install" in line]
print("startup pip commands:\n  " + "\n  ".join(pip_lines))

image: python:3.12-slim
startup pip commands:
  PIP_DISABLE_PIP_VERSION_CHECK=1 python3 -m pip install --quiet --no-warn-script-location 'scikit-learn==1.9.1' 'pandas>=2.2'  &&  python3 -m pip install --quiet --no-warn-script-location 'kfp==2.17.0' '--no-deps' 'typing-extensions>=3.7.4,<5; python_version<"3.9"' && "$0" "$@"


### ✍️ Your Turn

Write a component `fahrenheit(celsius: float) -> float` (use `**LOCAL`), run it locally with `run_local(fahrenheit, celsius=37.0)`, and store the task's `.output` in `body_temp_f`.

In [7]:
body_temp_f = None  # TODO: define the component, run it, take .output
check("body_temp_f", body_temp_f, 98.6, hint="F = C × 9/5 + 32; decorate with @dsl.component(**LOCAL).")

⏳ body_temp_f: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
@dsl.component(**LOCAL)
def fahrenheit(celsius: float) -> float:
    return celsius * 9 / 5 + 32


body_temp_f = run_local(fahrenheit, celsius=37.0)[0].output
check("body_temp_f", body_temp_f, 98.6)
```
</details>

> 💡 **Interview angle:** "`packages_to_install` or a custom image?" — `packages_to_install` is great for prototyping, but it pip-installs on *every* task start (slow, and depends on PyPI being reachable and versions staying available). Production components use a pinned, pre-built image (`base_image=` or `@dsl.container_component`).

## 4. Parameters vs Artifacts 🟡

| | **Parameters** | **Artifacts** |
|---|---|---|
| What | small values: `int`, `float`, `str`, `bool`, `list`, `dict` | files/directories + metadata: `Dataset`, `Model`, `Metrics`, `ClassificationMetrics`, `HTML`, `Markdown`, `Artifact` |
| Stored | inline in the run's metadata | in the **pipeline root** (object storage: `gs://`, `s3://`, `minio://`; a local folder here) |
| In a component | normal arguments / return value | `Output[Model]` → write to `model.path`; `Input[Dataset]` → read from `data.path`; add `.metadata` |
| Use for | hyperparameters, thresholds, IDs, a metric value you branch on | datasets, trained models, reports, metrics you want visualized and tracked |

`Metrics` and `ClassificationMetrics` are special: you call `metrics.log_metric("auc", 0.97)` or `cm.log_confusion_matrix(...)` and the UI renders them.

Let's write a real dataset artifact — the **breast cancer** diagnostic dataset bundled with scikit-learn — and read it back.

In [8]:
@dsl.component(**LOCAL)
def load_breast_cancer_data(train_set: Output[Dataset], test_set: Output[Dataset], test_size: float = 0.25, seed: int = 0) -> int:
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split
    frame = load_breast_cancer(as_frame=True).frame
    train, test = train_test_split(frame, test_size=test_size, random_state=seed, stratify=frame["target"])
    train.to_csv(train_set.path, index=False)
    test.to_csv(test_set.path, index=False)
    for artifact, part in ((train_set, train), (test_set, test)):
        artifact.metadata.update({"rows": len(part), "positive_rate": round(float(part["target"].mean()), 3)})
    return frame.shape[1] - 1                      # number of features, as a parameter


data_task, seconds, _ = run_local(load_breast_cancer_data)
print("parameter output (n_features):", data_task.outputs["Output"])
for name in ("train_set", "test_set"):
    artifact = data_task.outputs[name]
    print(f"{name}: type={type(artifact).__name__} metadata={artifact.metadata} path={rel(artifact.path)}")

import pandas as pd

train_preview = pd.read_csv(data_task.outputs["train_set"].path)
print("\nread back:", train_preview.shape, "| first columns:", list(train_preview.columns[:4]))

parameter output (n_features): 30
train_set: type=Dataset metadata={'rows': 426.0, 'positive_rate': 0.627} path=$PIPELINE_ROOT/load-breast-cancer-data-2026-09-15-10-49-56-339836/load-breast-cancer-data/train_set
test_set: type=Dataset metadata={'rows': 143.0, 'positive_rate': 0.629} path=$PIPELINE_ROOT/load-breast-cancer-data-2026-09-15-10-49-56-339836/load-breast-cancer-data/test_set

read back: (426, 31) | first columns: ['mean radius', 'mean texture', 'mean perimeter', 'mean area']


### ✍️ Your Turn

For each output, decide **parameter** or **artifact**, and build `io_choices = {"learning_rate": ..., "trained_model": ..., "val_auc_to_branch_on": ..., "shap_report_html": ...}` with the values `"parameter"` or `"artifact"`.

In [9]:
io_choices = None  # TODO: a dict
check("io_choices", io_choices,
      {"learning_rate": "parameter", "trained_model": "artifact", "val_auc_to_branch_on": "parameter", "shap_report_html": "artifact"},
      hint="Small values you pass or branch on → parameter; files → artifact.")

⏳ io_choices: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
io_choices = {"learning_rate": "parameter", "trained_model": "artifact",
              "val_auc_to_branch_on": "parameter", "shap_report_html": "artifact"}
check("io_choices", io_choices,
      {"learning_rate": "parameter", "trained_model": "artifact", "val_auc_to_branch_on": "parameter", "shap_report_html": "artifact"})
```

A common pattern does both: log the AUC into a `Metrics` artifact for tracking **and** return it as a `float` parameter so `dsl.If` can branch on it.
</details>

> 💡 **Interview angle:** "Parameter or artifact?" — parameters are small, inline values (and can drive control flow); artifacts are files in object storage with metadata and lineage. Never push a dataset through a string parameter.

## 5. Building and Running a Pipeline Locally 🟡

A **pipeline** is a function decorated with `@dsl.pipeline` that *wires components together*:

- Calling a component inside a pipeline creates a **task** (it doesn't run yet).
- `task.output` is the single return value; `task.outputs["name"]` picks a named output or artifact. Passing it into another component creates the dependency **and** the data flow.
- Pipeline arguments become **pipeline parameters** you can change per run.
- Task settings like `.set_cpu_request("1")`, `.set_memory_limit("2Gi")`, `.set_retry(num_retries=2)`, `.set_display_name(...)` go into the compiled spec for the cluster.

```
  load-breast-cancer-data ──train_set──► train-logreg ──model──► evaluate-model ──► AUC (pipeline output)
           └──────────────────────────────test_set───────────────────┘
```

In [10]:
@dsl.component(**LOCAL)
def train_logreg(train_set: Input[Dataset], model: Output[Model], C: float = 1.0):
    import joblib
    import pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    df = pd.read_csv(train_set.path)
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    clf.fit(df.drop(columns="target"), df["target"])
    joblib.dump(clf, model.path)
    model.metadata.update({"framework": "scikit-learn", "estimator": "LogisticRegression", "C": C})


@dsl.component(**LOCAL)
def evaluate_model(test_set: Input[Dataset], model: Input[Model], metrics: Output[Metrics],
                   confusion: Output[ClassificationMetrics]) -> float:
    import joblib
    import pandas as pd
    from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

    df = pd.read_csv(test_set.path)
    X, y = df.drop(columns="target"), df["target"]
    clf = joblib.load(model.path)
    auc = float(roc_auc_score(y, clf.predict_proba(X)[:, 1]))
    metrics.log_metric("auc", round(auc, 4))
    metrics.log_metric("accuracy", round(float(accuracy_score(y, clf.predict(X))), 4))
    confusion.log_confusion_matrix(["malignant", "benign"], confusion_matrix(y, clf.predict(X)).tolist())
    return auc


@dsl.pipeline(name="breast-cancer-train", description="Load data, train logistic regression, evaluate on a held-out split")
def train_pipeline(C: float = 1.0, test_size: float = 0.25) -> float:
    data = load_breast_cancer_data(test_size=test_size)
    trained = train_logreg(train_set=data.outputs["train_set"], C=C)
    trained.set_display_name("train logistic regression").set_cpu_request("1").set_memory_limit("2Gi")
    scored = evaluate_model(test_set=data.outputs["test_set"], model=trained.outputs["model"])
    return scored.outputs["Output"]              # evaluate_model also has artifact outputs, so name the parameter


run, seconds, train_log = run_local(train_pipeline, C=1.0)
print(f"pipeline output (test AUC): {run.output:.4f} — finished in {seconds:.1f} s")
print("task order and status:", task_statuses(train_log))

ℹ️ local runner: Task 'train-logreg': settings ['display_name', 'memory_limit', 'cpu_request'] have no effect in the current local runner and will be ignored.
pipeline output (test AUC): 0.9952 — finished in 3.2 s
task order and status: [('load-breast-cancer-data', 'SUCCESS'), ('train-logreg', 'SUCCESS'), ('evaluate-model', 'SUCCESS')]


Every task gets a folder under the pipeline root. Artifact files live there, and each task's `executor_output.json` records its output parameters and artifact **metadata** (that's where `Metrics` values live):

In [11]:
def show_tree(folder, max_depth=3):
    folder = Path(folder)
    print(rel(folder) + "/")
    for path in sorted(folder.rglob("*")):
        depth = len(path.relative_to(folder).parts)
        if depth <= max_depth:
            print("    " * depth + path.name + ("/" if path.is_dir() else f"  ({path.stat().st_size:,} bytes)"))


def executor_output(run_dir, task_name):
    return json.loads((Path(run_dir) / task_name / "executor_output.json").read_text())


def artifact_metadata(run_dir, task_name, artifact_name):
    return executor_output(run_dir, task_name)["artifacts"][artifact_name]["artifacts"][0]["metadata"]


train_run_dir = latest_run_dir("breast-cancer-train")
show_tree(train_run_dir)
print("\nmetrics     :", artifact_metadata(train_run_dir, "evaluate-model", "metrics"))
print("confusion   :", artifact_metadata(train_run_dir, "evaluate-model", "confusion")["confusionMatrix"]["rows"])
print("model meta  :", artifact_metadata(train_run_dir, "train-logreg", "model"))

$PIPELINE_ROOT/breast-cancer-train-2026-09-15-10-49-58-543345/
    evaluate-model/
        executor_output.json  (767 bytes)
    load-breast-cancer-data/
        executor_output.json  (667 bytes)
        test_set  (30,828 bytes)
        train_set  (91,049 bytes)
    train-logreg/
        executor_output.json  (339 bytes)
        model  (3,105 bytes)

metrics     : {'auc': 0.9952, 'accuracy': 0.958}
confusion   : [{'row': [50, 3]}, {'row': [3, 87]}]
model meta  : {'framework': 'scikit-learn', 'estimator': 'LogisticRegression', 'C': 1.0}


> 💡 **Interview angle:** "How does data move between pipeline steps?" — parameters are passed by value in the run metadata; artifacts are written by the producing step to object storage under the pipeline root, and the consuming step receives a local path (or URI) to read. Steps never share memory or a filesystem implicitly.

## 6. Control Flow: If, ParallelFor, ExitHandler 🔴

| Construct | Meaning | Airflow analogue |
|---|---|---|
| `with dsl.If(task.output >= 0.9):` · `dsl.Elif(...)` · `dsl.Else()` | run the enclosed tasks only when the condition on a **parameter** holds | `@task.branch` |
| `dsl.OneOf(a.output, b.output)` | the output of whichever branch ran | join after branch |
| `with dsl.ParallelFor(items, parallelism=2) as item:` | fan out one task set per item (items can come from another task) | dynamic task mapping `.expand()` |
| `dsl.Collected(task.output)` | fan in: the list of all loop outputs | reducing a mapped task's results |
| `with dsl.ExitHandler(exit_task):` | run `exit_task` when the enclosed tasks finish, **even if they fail**; it can receive `dsl.PipelineTaskFinalStatus` | `trigger_rule="all_done"` cleanup/alert |

Below we tune the regularization strength `C` with 5-fold cross-validation **on the training split only** (one loop iteration per value), pick the best, and branch on its score.

In [12]:
@dsl.component(**LOCAL)
def cross_validate_logreg(train_set: Input[Dataset], C: float) -> dict:
    import pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    df = pd.read_csv(train_set.path)
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    scores = cross_val_score(clf, df.drop(columns="target"), df["target"], cv=5, scoring="roc_auc")
    return {"C": C, "cv_auc": round(float(scores.mean()), 5)}


@dsl.component(**LOCAL)
def pick_best(results: List[dict]) -> NamedTuple("Best", [("C", float), ("cv_auc", float)]):
    best = max(results, key=lambda r: r["cv_auc"])
    print(f"candidates: {results}")
    return (float(best["C"]), float(best["cv_auc"]))


@dsl.component(**LOCAL)
def decision(message: str, cv_auc: float) -> str:
    return f"{message} (cv_auc={cv_auc:.4f})"


@dsl.pipeline(name="tune-and-gate")
def tune_and_gate(min_auc: float = 0.99, review_auc: float = 0.95) -> str:
    data = load_breast_cancer_data()
    with dsl.ParallelFor(items=[0.01, 0.1, 1.0, 10.0], parallelism=2) as C:
        cv = cross_validate_logreg(train_set=data.outputs["train_set"], C=C)
    best = pick_best(results=dsl.Collected(cv.output))
    with dsl.If(best.outputs["cv_auc"] >= min_auc, name="ship"):
        ship = decision(message="ship it", cv_auc=best.outputs["cv_auc"])
    with dsl.Elif(best.outputs["cv_auc"] >= review_auc, name="review"):
        review = decision(message="needs human review", cv_auc=best.outputs["cv_auc"])
    with dsl.Else(name="reject"):
        reject = decision(message="do not ship", cv_auc=best.outputs["cv_auc"])
    return dsl.OneOf(ship.output, review.output, reject.output)


gate_run, seconds, gate_log = run_local(tune_and_gate)
gate_dir = latest_run_dir("tune-and-gate")
best = executor_output(gate_dir, "pick-best")["parameterValues"]
print(f"best C = {best['C']} with CV AUC {best['cv_auc']:.4f} → pipeline output: {gate_run.output!r} ({seconds:.1f} s)")
print("task folders created:", sorted(p.name for p in gate_dir.iterdir()))
ran_decisions = [name for name, status in task_statuses(gate_log) if name.startswith("decision")]
print("decision tasks that actually ran:", ran_decisions, "(the other branches never started)")

best C = 0.1 with CV AUC 0.9959 → pipeline output: 'ship it (cv_auc=0.9959)' (3.4 s)
task folders created: ['decision', 'for-loop-2-iteration-0', 'for-loop-2-iteration-1', 'for-loop-2-iteration-2', 'for-loop-2-iteration-3', 'load-breast-cancer-data', 'pick-best']
decision tasks that actually ran: ['decision'] (the other branches never started)


An **exit handler** runs even when a step fails — ideal for cleanup and notifications. Here a step fails on purpose:

In [13]:
@dsl.component(**LOCAL)
def flaky_upload(rows: int):
    raise ConnectionError(f"object store unavailable while uploading {rows} rows")


@dsl.component(**LOCAL)
def notify_on_exit(status: dsl.PipelineTaskFinalStatus) -> str:
    message = f"pipeline {status.pipeline_task_name} finished with state {status.state}"
    print(message)
    return message


@dsl.pipeline(name="exit-handler-demo")
def exit_handler_demo():
    with dsl.ExitHandler(notify_on_exit(), name="upload-block"):
        flaky_upload(rows=569)


exit_run, _, exit_log = run_local(exit_handler_demo)
print("statuses:", task_statuses(exit_log))
exit_dir = latest_run_dir("exit-handler-demo")
print("exit task output:", executor_output(exit_dir, "notify-on-exit")["parameterValues"]["Output"])

statuses: [('flaky-upload', 'FAILURE'), ('notify-on-exit', 'SUCCESS')]
exit task output: pipeline exit-handler-1 finished with state FAILED


### ✍️ Your Turn

Run `tune_and_gate` with `min_auc=0.9999` and `review_auc=0.5`, and store the pipeline's output string in `strict_gate_message`. (The checker only looks at the start of the message.)

In [14]:
strict_gate_message = None  # TODO: run_local(tune_and_gate, ...)[0].output
check("strict_gate_message", None if strict_gate_message is None else strict_gate_message.split(" (")[0], "needs human review",
      hint="The best CV AUC is below 0.9999 but above 0.5, so the Elif branch runs.")

⏳ strict_gate_message: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
strict_gate_message = run_local(tune_and_gate, min_auc=0.9999, review_auc=0.5)[0].output
check("strict_gate_message", strict_gate_message.split(" (")[0], "needs human review")
```
</details>

> 💡 **Interview angle:** "How do you gate model deployment in a pipeline?" — compute the metric as a *parameter* output, then `dsl.If(metric >= threshold)` around the register/deploy step (with `Elif`/`Else` for review or rejection), and an `ExitHandler` for notifications that must fire even on failure.

## 7. Caching 🟡

If a task runs again with the **same component (image + command) and the same inputs**, KFP can reuse the previous outputs instead of re-executing. On a cluster this is on by default per task; locally it's on because we passed `enable_caching=True` to `local.init`.

- Change a pipeline parameter → only tasks whose inputs changed (and their downstream tasks) re-run.
- Turn caching off for steps that aren't deterministic functions of their inputs (pulling "the latest" data, calling an external API): `task.set_caching_options(enable_caching=False)`, or for a whole cluster run `client.create_run_from_pipeline_func(..., enable_caching=False)`.

In [15]:
def cache_hits(log_text):
    return sorted(set(re.findall(r"Task ([\w-]+) cache hit", log_text)))


for label, kwargs in (("same params again", {"C": 1.0}), ("C changed to 0.1", {"C": 0.1})):
    cached_run, seconds, cached_log = run_local(train_pipeline, show_local_notes=False, **kwargs)
    executed = [name for name, _ in task_statuses(cached_log)]
    print(f"{label:18s} → {seconds:5.1f} s | cache hits: {cache_hits(cached_log)} | executed: {executed} | AUC {cached_run.output:.4f}")
print("cache entries on disk:", len(list(CACHE_ROOT.glob("*.json"))))

same params again  →   1.0 s | cache hits: ['load-breast-cancer-data', 'train-logreg'] | executed: ['evaluate-model'] | AUC 0.9952


C changed to 0.1   →   2.5 s | cache hits: ['load-breast-cancer-data'] | executed: ['train-logreg', 'evaluate-model'] | AUC 0.9929
cache entries on disk: 13


Read the two lines carefully:

- **Same parameters:** data loading and training were cache hits. `evaluate-model` still executed: the local cache only reuses an entry whose output artifact files still exist on disk, and `Metrics`/`ClassificationMetrics` artifacts are metadata-only (no file), so locally that step always re-runs. It's cheap here, and on a cluster the backend's cache handles such steps itself.
- **`C` changed:** only `load-breast-cancer-data` was reused. `C` feeds `train-logreg`, and its new model feeds `evaluate-model`, so both re-ran — a changed upstream output changes every downstream cache key.

> 💡 **Interview angle:** "How does pipeline caching decide to skip a step, and when is it dangerous?" — the cache key is the component definition plus its input values/artifacts; identical key → reuse outputs. It's dangerous for non-deterministic steps (reading "today's" data, random seeds not passed as inputs, external side effects) — disable caching there.

## 8. Compiling to IR YAML and Reading It 🟡

`compiler.Compiler().compile(pipeline, "pipeline.yaml")` produces the **pipeline spec** (the "IR" — intermediate representation). This one YAML file is what you upload to the Kubeflow UI, submit with `kfp.Client`, or run on Vertex AI Pipelines. Its main sections:

| Key | Contains |
|---|---|
| `pipelineInfo`, `sdkVersion`, `schemaVersion` | name and versions |
| `root` | the top-level DAG: `tasks`, pipeline `inputDefinitions` (parameters) and `outputDefinitions` |
| `components` | every component's interface (and nested DAGs for `If` branches and loops) |
| `deploymentSpec.executors` | how to run each component: `image`, `command`, `args`, `resources` |

Inside `root.dag.tasks`, each task lists `componentRef`, `dependentTasks`, `inputs` (constants, pipeline parameters, or `taskOutputArtifact`/`taskOutputParameter` from a producer task), plus `triggerPolicy` for conditions and `iteratorPolicy` for loops.

In [16]:
compiler.Compiler().compile(tune_and_gate, str(SPECS / "tune_and_gate.yaml"))
ir = yaml.safe_load((SPECS / "tune_and_gate.yaml").read_text())
print("top-level keys:", sorted(ir))
print("pipeline:", ir["pipelineInfo"]["name"], "| sdk:", ir["sdkVersion"], "| size:", f"{(SPECS / 'tune_and_gate.yaml').stat().st_size:,} bytes")
print("pipeline parameters:", {k: v["parameterType"] for k, v in ir["root"]["inputDefinitions"]["parameters"].items()})
print("root tasks:", sorted(ir["root"]["dag"]["tasks"]))
print("components:", sorted(ir["components"]))
print("executor images:", {name: ex["container"]["image"] for name, ex in ir["deploymentSpec"]["executors"].items()})


def find_key(node, key, path=""):
    """Yield (path, value) for every occurrence of `key` in a nested dict/list."""
    if isinstance(node, dict):
        for k, v in node.items():
            if k == key:
                yield path + "/" + k, v
            yield from find_key(v, key, path + "/" + k)
    elif isinstance(node, list):
        for i, v in enumerate(node):
            yield from find_key(v, key, f"{path}[{i}]")


for where, policy in find_key(ir, "iteratorPolicy"):
    print("loop:", where.split("/tasks/")[-1], policy)
for where, policy in find_key(ir, "triggerPolicy"):
    print("condition on", where.split("/tasks/")[-2].split("/")[0] + ":", policy["condition"])

top-level keys: ['components', 'deploymentSpec', 'pipelineInfo', 'root', 'schemaVersion', 'sdkVersion']
pipeline: tune-and-gate | sdk: kfp-2.17.0 | size: 18,454 bytes
pipeline parameters: {'min_auc': 'NUMBER_DOUBLE', 'review_auc': 'NUMBER_DOUBLE'}
root tasks: ['condition-branches-3', 'for-loop-2', 'load-breast-cancer-data', 'pick-best']
components: ['comp-condition-4', 'comp-condition-5', 'comp-condition-6', 'comp-condition-branches-3', 'comp-cross-validate-logreg', 'comp-decision', 'comp-decision-2', 'comp-decision-3', 'comp-for-loop-2', 'comp-load-breast-cancer-data', 'comp-pick-best']
executor images: {'exec-cross-validate-logreg': 'python:3.12', 'exec-decision': 'python:3.12', 'exec-decision-2': 'python:3.12', 'exec-decision-3': 'python:3.12', 'exec-load-breast-cancer-data': 'python:3.12', 'exec-pick-best': 'python:3.12'}
loop: for-loop-2/iteratorPolicy {'parallelismLimit': 2}
condition on : inputs.parameter_values['pipelinechannel--pick-best-cv_auc'] >= inputs.parameter_values['pi

### ✍️ Your Turn

Using `find_key`, get the **`parallelismLimit`** of the `ParallelFor` loop in `ir` and store it in `loop_parallelism`.

In [17]:
loop_parallelism = None  # TODO
check("loop_parallelism", loop_parallelism, 2, hint="next(find_key(ir, 'iteratorPolicy'))[1]['parallelismLimit']")

⏳ loop_parallelism: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
loop_parallelism = next(find_key(ir, "iteratorPolicy"))[1]["parallelismLimit"]
check("loop_parallelism", loop_parallelism, 2)
```
</details>

> 💡 **Interview angle:** "What does the KFP compiler produce, and why does it matter?" — a platform-neutral pipeline spec (protobuf serialized as YAML) describing the DAG, component interfaces and container executors. Because it's decoupled from the SDK, the same file runs on Kubeflow Pipelines or Vertex AI Pipelines, can be versioned in git, and can be diffed in code review.

## 9. Katib: Hyperparameter Tuning Experiments 🟡

**Katib** runs hyperparameter search as a Kubernetes resource called an **`Experiment`**. You declare *what* to optimize; Katib launches **trials** (each one a pod/job with specific hyperparameters), reads the metric each trial reports, and proposes new values.

| Field | Meaning |
|---|---|
| `objective` | `type` (maximize/minimize), `objectiveMetricName`, optional `goal` and `additionalMetricNames` |
| `algorithm.algorithmName` | `random`, `grid`, `bayesianoptimization`, `tpe`, `multivariate-tpe`, `cmaes`, `hyperband`, … |
| `parallelTrialCount` / `maxTrialCount` / `maxFailedTrialCount` | how many trials run at once / in total / may fail before the experiment fails |
| `parameters` | search space: `name`, `parameterType` (`int`, `double`, `categorical`, `discrete`), `feasibleSpace` (`min`/`max`/`step` or `list`) |
| `trialTemplate` | the Job each trial runs; `trialParameters` map search-space names into the command via `${trialParameters.<name>}` |
| `metricsCollectorSpec` | **how Katib reads the metric** (see below) |

**The metrics collector — a classic source of failed experiments.** The default collector kind is **`StdOut`**: a sidecar reads the training container's standard output and parses lines with the default regex `([\w|-]+)\s*=\s*([+-]?\d*(\.\d+)?([Ee][+-]?\d+)?)` — i.e. **`name=value`** lines such as `accuracy=0.953`. Printing JSON to stdout does **not** match that format. JSON *is* supported by the **`File`** collector with `source.fileSystemPath.format: JSON` (one JSON object per line), and you can also supply a custom `source.filter.metricsFormat` regex.

Here's a valid experiment for a **random forest** — note the parameters are ones `RandomForestClassifier` really has (the old version of this notebook tuned a `learning_rate`, which random forests don't have):

In [18]:
KATIB_EXPERIMENT = yaml.safe_load("""
apiVersion: kubeflow.org/v1beta1
kind: Experiment
metadata:
  name: breast-cancer-random-forest
  namespace: kubeflow-user-example-com
spec:
  objective:
    type: maximize
    goal: 0.99
    objectiveMetricName: roc_auc
    additionalMetricNames: [accuracy]
  algorithm:
    algorithmName: bayesianoptimization
  parallelTrialCount: 3
  maxTrialCount: 24
  maxFailedTrialCount: 3
  metricsCollectorSpec:
    collector:
      kind: StdOut
  parameters:
    - name: n_estimators
      parameterType: int
      feasibleSpace: {min: "50", max: "400", step: "50"}
    - name: max_depth
      parameterType: int
      feasibleSpace: {min: "2", max: "16"}
    - name: min_samples_leaf
      parameterType: int
      feasibleSpace: {min: "1", max: "10"}
    - name: max_features
      parameterType: categorical
      feasibleSpace: {list: ["sqrt", "log2"]}
  trialTemplate:
    primaryContainerName: training-container
    trialParameters:
      - {name: nEstimators, reference: n_estimators, description: number of trees}
      - {name: maxDepth, reference: max_depth, description: maximum tree depth}
      - {name: minSamplesLeaf, reference: min_samples_leaf, description: minimum samples per leaf}
      - {name: maxFeatures, reference: max_features, description: features considered per split}
    trialSpec:
      apiVersion: batch/v1
      kind: Job
      spec:
        template:
          spec:
            restartPolicy: Never
            containers:
              - name: training-container
                image: ghcr.io/example/rf-trainer:1.0.0
                command:
                  - python
                  - train.py
                  - --n_estimators=${trialParameters.nEstimators}
                  - --max_depth=${trialParameters.maxDepth}
                  - --min_samples_leaf=${trialParameters.minSamplesLeaf}
                  - --max_features=${trialParameters.maxFeatures}
""")

from sklearn.ensemble import RandomForestClassifier

spec = KATIB_EXPERIMENT["spec"]
search_space = {p["name"] for p in spec["parameters"]}
valid_rf_params = set(RandomForestClassifier().get_params())
print("search space:", sorted(search_space))
print("all are real RandomForestClassifier parameters:", search_space <= valid_rf_params)
print("'learning_rate' is a RandomForestClassifier parameter:", "learning_rate" in valid_rf_params)
references = {tp["reference"] for tp in spec["trialTemplate"]["trialParameters"]}
command = " ".join(spec["trialTemplate"]["trialSpec"]["spec"]["template"]["spec"]["containers"][0]["command"])
used = set(re.findall(r"\$\{trialParameters\.(\w+)\}", command))
print("every trialParameter references a search-space parameter:", references == search_space)
print("every trialParameter is used in the command:", used == {tp["name"] for tp in spec["trialTemplate"]["trialParameters"]})

search space: ['max_depth', 'max_features', 'min_samples_leaf', 'n_estimators']
all are real RandomForestClassifier parameters: True
'learning_rate' is a RandomForestClassifier parameter: False
every trialParameter references a search-space parameter: True
every trialParameter is used in the command: True


Now let's run **one trial's training code** locally (real data, real model) and parse its output exactly the way Katib's default `StdOut` collector would — once with `name=value` lines, once with JSON:

In [19]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

KATIB_DEFAULT_METRICS_FORMAT = r"([\w|-]+)\s*=\s*([+-]?\d*(\.\d+)?([Ee][+-]?\d+)?)"


def katib_stdout_metrics(stdout, wanted):
    """Latest value of each wanted metric found by Katib's default StdOut regex."""
    found = {}
    for line in stdout.splitlines():
        for name, value, *_ in re.findall(KATIB_DEFAULT_METRICS_FORMAT, line):
            if name in wanted and value not in ("", "+", "-"):
                found[name] = float(value)
    return found


def trial(n_estimators, max_depth, min_samples_leaf, max_features, style):
    """What train.py would do inside one trial pod."""
    from sklearn.metrics import accuracy_score, roc_auc_score
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
    rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                                max_features=max_features, random_state=0, n_jobs=2).fit(X_train, y_train)
    auc, acc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1]), accuracy_score(y_val, rf.predict(X_val))
    if style == "name=value":
        return f"epoch 1 done\nroc_auc={auc:.4f}\naccuracy={acc:.4f}\n"
    return json.dumps({"roc_auc": round(auc, 4), "accuracy": round(acc, 4)}) + "\n"


wanted = {spec["objective"]["objectiveMetricName"], *spec["objective"]["additionalMetricNames"]}
for style in ("name=value", "json"):
    stdout = trial(n_estimators=200, max_depth=8, min_samples_leaf=2, max_features="sqrt", style=style)
    print(f"{style:10s} stdout: {stdout.strip()!r}")
    print(f"{'':10s} Katib StdOut collector sees: {katib_stdout_metrics(stdout, wanted) or 'no metrics → trial reported as failed/metrics unavailable'}")

name=value stdout: 'epoch 1 done\nroc_auc=0.9761\naccuracy=0.9441'
           Katib StdOut collector sees: {'roc_auc': 0.9761, 'accuracy': 0.9441}


json       stdout: '{"roc_auc": 0.9761, "accuracy": 0.9441}'
           Katib StdOut collector sees: no metrics → trial reported as failed/metrics unavailable


### ✍️ Your Turn

Katib needs a `parameterType` for `class_weight`, which can be `"balanced"` or `None` (written as the string `"none"` in the YAML). Which type do you use? Store the string in `class_weight_type`.

In [20]:
class_weight_type = None  # TODO: "int", "double", "categorical" or "discrete"
check("class_weight_type", class_weight_type, "categorical", hint="Unordered string choices listed explicitly.")

⏳ class_weight_type: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class_weight_type = "categorical"
check("class_weight_type", class_weight_type, "categorical")
```

`discrete` is for an explicit list of *numeric* values (e.g. batch sizes 16, 32, 64); `categorical` is for strings. Your training script then maps `"none"` back to Python `None`.
</details>

> 💡 **Interview angle:** "Your Katib trials all show 'metrics not available' — why?" — the default `StdOut` collector only parses `name=value` lines matching its regex; the script printed JSON, printed a different metric name than `objectiveMetricName`, or wrote to a file without configuring the `File` collector. Fix the print format, the name, or the `metricsCollectorSpec`.

## 10. KServe: Serving with an InferenceService 🟡

**KServe** turns a model in object storage into an autoscaling HTTP/gRPC endpoint with one Kubernetes resource, the **`InferenceService`**:

- `spec.predictor.model.modelFormat.name: sklearn` picks a serving runtime that can load `.joblib`/`.pkl` files (`kserve-sklearnserver`).
- `storageUri` points to the **directory** containing the model file (`gs://`, `s3://`, `pvc://`, …).
- `protocolVersion: v2` uses the **Open Inference Protocol**: `POST /v2/models/<name>/infer` with a JSON body of named, typed tensors.
- `minReplicas`/`maxReplicas` control autoscaling (`minReplicas: 0` enables scale-to-zero in serverless mode), `resources` the pod size, and `canaryTrafficPercent` gradual rollouts.

In [21]:
INFERENCE_SERVICE = yaml.safe_load("""
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: breast-cancer-logreg
  namespace: kubeflow-user-example-com
spec:
  predictor:
    minReplicas: 1
    maxReplicas: 3
    model:
      modelFormat:
        name: sklearn
      protocolVersion: v2
      runtime: kserve-sklearnserver
      storageUri: gs://example-ml-models/breast-cancer/logreg/v3
      resources:
        requests: {cpu: "500m", memory: 1Gi}
        limits: {cpu: "1", memory: 2Gi}
""")
model_spec = INFERENCE_SERVICE["spec"]["predictor"]["model"]
print("kind:", INFERENCE_SERVICE["kind"], "| format:", model_spec["modelFormat"]["name"], "| protocol:", model_spec["protocolVersion"],
      "| storage:", model_spec["storageUri"])
print("endpoint path:", f"/v2/models/{INFERENCE_SERVICE['metadata']['name']}/infer")

kind: InferenceService | format: sklearn | protocol: v2 | storage: gs://example-ml-models/breast-cancer/logreg/v3
endpoint path: /v2/models/breast-cancer-logreg/infer


Without a cluster we can't call KServe, but we can **build the exact v2 request** a client would send and emulate what the sklearn server does with it (decode the tensor, call `predict`, encode the response), using the model trained by our pipeline in section 5:

In [22]:
import joblib
import numpy as np

served_model = joblib.load(train_run_dir / "train-logreg" / "model")
test_rows = pd.read_csv(train_run_dir / "load-breast-cancer-data" / "test_set").drop(columns="target").head(3)

request = {"inputs": [{"name": "input-0", "shape": list(test_rows.shape), "datatype": "FP64",
                       "data": test_rows.to_numpy().tolist()}]}
print("request body:", json.dumps(request)[:160], "…", f"({len(json.dumps(request)):,} bytes)")

tensor = request["inputs"][0]
features = np.asarray(tensor["data"], dtype=np.float64).reshape(tensor["shape"])       # server side: decode
predictions = served_model.predict(pd.DataFrame(features, columns=test_rows.columns))  # server side: predict
response = {"model_name": INFERENCE_SERVICE["metadata"]["name"],
            "outputs": [{"name": "predict", "shape": [len(predictions)], "datatype": "INT64", "data": predictions.tolist()}]}
print("response body:", json.dumps(response))

if shutil.which("kubectl") and os.environ.get("KUBECONFIG"):
    print("kubectl found — apply with: kubectl apply -f inference_service.yaml")
else:
    print("⏭️ Skipped deploying: no Kubernetes cluster configured (kubectl + KUBECONFIG). The YAML above is what you'd apply.")

request body: {"inputs": [{"name": "input-0", "shape": [3, 30], "datatype": "FP64", "data": [[13.68, 16.33, 87.76, 575.5, 0.09277, 0.07255, 0.01752, 0.0188, 0.1631, 0.06155,  … (813 bytes)
response body: {"model_name": "breast-cancer-logreg", "outputs": [{"name": "predict", "shape": [3], "datatype": "INT64", "data": [1, 0, 0]}]}
⏭️ Skipped deploying: no Kubernetes cluster configured (kubectl + KUBECONFIG). The YAML above is what you'd apply.


> 💡 **Interview angle:** "How would you deploy this model on Kubernetes with autoscaling and a safe rollout?" — push the artifact to object storage, create an `InferenceService` with the right `modelFormat`/runtime and resources, set min/max replicas, and roll out new versions with `canaryTrafficPercent` while watching latency and error metrics.

## 11. Running on a Cluster, and KFP vs Airflow 🟢

The pipelines above run unchanged on a real backend:

```python
client = kfp.Client(host="https://<your-kfp-endpoint>")                 # Kubeflow Pipelines
client.create_run_from_pipeline_func(tune_and_gate, arguments={"min_auc": 0.99}, experiment_name="gates")
# or upload SPECS/"tune_and_gate.yaml" in the UI, or submit it to Vertex AI Pipelines as a PipelineJob template
```

On a cluster, remember to remove `install_kfp_package=False` (or use an image that already contains `kfp`) and set `pipeline_root` to object storage.

In [23]:
if KFP_HOST:
    client = kfp.Client(host=KFP_HOST)
    cluster_run = client.create_run_from_pipeline_func(tune_and_gate, arguments={"min_auc": 0.99}, experiment_name="notebook-demo")
    print("submitted run:", cluster_run.run_id)
else:
    print("⏭️ Skipped cluster submission: set KFP_HOST to a Kubeflow Pipelines endpoint to submit these pipelines for real.")

if DOCKER_DAEMON and DOCKER_SDK:
    local.init(runner=local.DockerRunner(), pipeline_root=str(PIPELINE_ROOT), raise_on_error=False)
    try:
        docker_task, seconds, _ = run_local(dsl.component(base_image="python:3.12")(add.python_func), a=1, b=2)
        print(f"DockerRunner: add(1, 2) → {docker_task.output} inside a python:3.12 container ({seconds:.1f} s)")
    finally:
        init_local()
else:
    missing = [what for what, ok in (("a running Docker daemon", DOCKER_DAEMON), ("the `docker` Python package (pip install docker)", DOCKER_SDK)) if not ok]
    print(f"⏭️ Skipped DockerRunner demo: needs {' and '.join(missing)}. With both, each task runs in its real container image.")

⏭️ Skipped cluster submission: set KFP_HOST to a Kubeflow Pipelines endpoint to submit these pipelines for real.


DockerRunner: add(1, 2) → 3 inside a python:3.12 container (2.5 s)


**KFP or Airflow?**

| Question | Lean **Kubeflow Pipelines / Vertex AI Pipelines** | Lean **Airflow** |
|---|---|---|
| What are the steps? | ML workloads: training, tuning, batch scoring, evaluation | any scheduled work: ETL, SQL, reports, API calls, *triggering* ML |
| Where do steps run? | each step in its own container on Kubernetes (GPUs, per-step resources) | Airflow workers (or pods via KubernetesExecutor / KubernetesPodOperator) |
| Data between steps | typed artifacts in object storage with lineage | small XComs; data lives in external storage |
| Scheduling | recurring runs exist, but time-based scheduling and backfills are basic | rich schedules, catchup/backfill, assets, sensors |
| Reproducibility features | compiled spec, caching keyed on inputs, artifact lineage | idempotent tasks keyed by logical date, DAG versioning |
| Team / infra | ML platform team already on Kubernetes or Google Cloud | data engineering team, many integrations needed |

Many companies use **both**: Airflow owns the business schedule and dependencies (data ready → trigger training), and a KFP/Vertex pipeline owns the ML steps.

> 💡 **Interview angle:** "Would you use Kubeflow Pipelines or Airflow for a nightly retraining job?" — if it's one step in a larger data workflow with many integrations and schedules, Airflow (possibly triggering a KFP/Vertex pipeline); if the job is a multi-step ML workload needing containers per step, GPUs, artifact lineage and caching on Kubernetes, KFP. Mention operational cost: KFP needs a Kubernetes cluster (or Vertex), Airflow needs its own components.

## 🔧 Build It From Scratch

The compiled YAML *is* the pipeline. Can we recover the graph from it without the SDK? We'll parse the IR with plain dictionaries and rebuild:

1. **Nodes** — every executable task, plus the *group* (loop / condition / exit handler) it sits in. Groups are components whose spec contains a nested `dag`.
2. **Control edges** — from `dependentTasks`.
3. **Data edges** — which producer output feeds which consumer input: `taskOutputArtifact` / `taskOutputParameter` give the producer directly; inside a group, `componentInputArtifact` / `componentInputParameter` point at the group's inputs, which we resolve through the parent scope.

Then we assert the result matches the pipeline we wrote in section 5, and that a topological order of the graph agrees with the order tasks actually finished in the local run.

In [24]:
def reconstruct(ir):
    nodes, control, data = {}, set(), set()

    def walk(dag_spec, scope, channels):
        for name, task_spec in dag_spec.get("tasks", {}).items():
            resolved = {}
            for kind in ("artifacts", "parameters"):
                for input_name, source in task_spec.get("inputs", {}).get(kind, {}).items():
                    ref = source.get("taskOutputArtifact") or source.get("taskOutputParameter")
                    if ref:
                        resolved[input_name] = (ref["producerTask"], ref.get("outputArtifactKey") or ref.get("outputParameterKey"))
                    key = source.get("componentInputArtifact") or source.get("componentInputParameter")
                    if key in channels:
                        resolved[input_name] = channels[key]
            for upstream in task_spec.get("dependentTasks", []):
                control.add((upstream, name))
            component = ir["components"][task_spec["componentRef"]["name"]]
            if "dag" in component:                                   # a group: recurse with its inputs as channels
                nodes[name] = {"kind": "group", "scope": tuple(scope)}
                walk(component["dag"], scope + [name], {**channels, **resolved})
            else:
                nodes[name] = {"kind": "task", "scope": tuple(scope)}
                for input_name, (producer, output_key) in resolved.items():
                    data.add((producer, output_key, name, input_name))

    walk(ir["root"]["dag"], [], {})
    return nodes, control, data


def topological_order(nodes, control):
    indegree = {n: 0 for n in nodes}
    for _, down in control:
        indegree[down] += 1
    ready, order = sorted(n for n, d in indegree.items() if d == 0), []
    while ready:
        node = ready.pop(0)
        order.append(node)
        for up, down in sorted(control):
            if up == node:
                indegree[down] -= 1
                if indegree[down] == 0:
                    ready.append(down)
    assert len(order) == len(nodes), "cycle in the compiled graph"
    return order


compiler.Compiler().compile(train_pipeline, str(SPECS / "train_pipeline.yaml"))
train_ir = yaml.safe_load((SPECS / "train_pipeline.yaml").read_text())
nodes, control, data = reconstruct(train_ir)

assert nodes == {name: {"kind": "task", "scope": ()} for name in ("load-breast-cancer-data", "train-logreg", "evaluate-model")}
assert control == {("load-breast-cancer-data", "train-logreg"), ("load-breast-cancer-data", "evaluate-model"),
                   ("train-logreg", "evaluate-model")}
assert data == {("load-breast-cancer-data", "train_set", "train-logreg", "train_set"),
                ("load-breast-cancer-data", "test_set", "evaluate-model", "test_set"),
                ("train-logreg", "model", "evaluate-model", "model")}
order = topological_order(nodes, control)
finished = [name for name, status in task_statuses(train_log) if status == "SUCCESS"]
assert order == finished, (order, finished)
print("✅ train_pipeline graph rebuilt from YAML matches the definition")
print("   order:", " → ".join(order))
for producer, key, consumer, input_name in sorted(data):
    print(f"   {producer}.{key} ──► {consumer}.{input_name}")

✅ train_pipeline graph rebuilt from YAML matches the definition
   order: load-breast-cancer-data → train-logreg → evaluate-model
   load-breast-cancer-data.test_set ──► evaluate-model.test_set
   load-breast-cancer-data.train_set ──► train-logreg.train_set
   train-logreg.model ──► evaluate-model.model


The same parser handles nested groups. For `tune_and_gate`, the loop body and the three branches live inside groups, and the branches' `cv_auc` input must be traced back through the condition group to `pick-best`:

In [25]:
gate_nodes, gate_control, gate_data = reconstruct(ir)
leaf_tasks = sorted(n for n, info in gate_nodes.items() if info["kind"] == "task")
print("tasks :", {n: "/".join(gate_nodes[n]["scope"]) or "(root)" for n in leaf_tasks})
print("groups:", sorted(n for n, info in gate_nodes.items() if info["kind"] == "group"))

loop_scope = gate_nodes["cross-validate-logreg"]["scope"]
assert len(loop_scope) == 1 and loop_scope[0].startswith("for-loop")
assert ("load-breast-cancer-data", "train_set", "cross-validate-logreg", "train_set") in gate_data
decision_tasks = [n for n in leaf_tasks if n.startswith("decision")]
assert len(decision_tasks) == 3 and all(gate_nodes[n]["scope"][0].startswith("condition-branches") for n in decision_tasks)
assert all(("pick-best", "cv_auc", n, "cv_auc") in gate_data for n in decision_tasks)
assert any(p.startswith("for-loop") and c == "pick-best" for p, _, c, _ in gate_data)
print("✅ loop body, fan-in to pick-best, and all three branches reading pick-best.cv_auc recovered from the YAML")

tasks : {'cross-validate-logreg': 'for-loop-2', 'decision': 'condition-branches-3/condition-4', 'decision-2': 'condition-branches-3/condition-5', 'decision-3': 'condition-branches-3/condition-6', 'load-breast-cancer-data': '(root)', 'pick-best': '(root)'}
groups: ['condition-4', 'condition-5', 'condition-6', 'condition-branches-3', 'for-loop-2']
✅ loop body, fan-in to pick-best, and all three branches reading pick-best.cv_auc recovered from the YAML


## ⚠️ Common Pitfalls

### ❌ Pitfall 1 — Using code defined outside the component

Only the function's own source is shipped to the task. Module-level helpers and imports don't exist there.

In [26]:
def normalize_name(text):                                  # lives in the notebook, NOT inside the component
    return text.strip().lower()


@dsl.component(**LOCAL)
def clean_label_wrong(label: str) -> str:
    return normalize_name(label)                          # ❌


@dsl.component(**LOCAL)
def clean_label_right(label: str) -> str:
    def normalize_name(text):                             # ✅ defined inside
        return text.strip().lower()
    return normalize_name(label)


_, _, wrong_log = run_local(clean_label_wrong, label="  Benign ")
print("❌", [line.strip() for line in wrong_log.splitlines() if "Error" in line][-1])
print("✅", repr(run_local(clean_label_right, label="  Benign ")[0].output))

❌ NameError: name 'normalize_name' is not defined


✅ 'benign'


### ❌ Pitfall 2 — `.output` on a task with several outputs

In [27]:
try:
    @dsl.pipeline(name="wrong-output")
    def wrong_output_pipeline() -> float:
        data = load_breast_cancer_data()
        trained = train_logreg(train_set=data.outputs["train_set"])
        return evaluate_model(test_set=data.outputs["test_set"], model=trained.outputs["model"]).output   # ❌
except AttributeError as err:
    print("❌ AttributeError while defining the pipeline:", err)

print("✅ name it instead: evaluate_model(...).outputs['Output'] — outputs available:",
      sorted(evaluate_model.component_spec.outputs))

❌ AttributeError while defining the pipeline: The task has multiple outputs. Please reference the output by its name.
✅ name it instead: evaluate_model(...).outputs['Output'] — outputs available: ['Output', 'confusion', 'metrics']


### ❌ Pitfall 3 — Trusting the type of fanned-in values

In local runs with KFP 2.17, integers collected from a `ParallelFor` arrive as JSON numbers that deserialize as floats, so a component annotated `-> int` that returns `sum(counts)` fails its own output type check.

In [28]:
@dsl.component(**LOCAL)
def rows_in_shard(shard: int) -> int:
    return 1000 + shard


@dsl.component(**LOCAL)
def total_rows_wrong(counts: List[int]) -> int:
    return sum(counts)                                    # ❌ may be a float


@dsl.component(**LOCAL)
def total_rows_right(counts: List[int]) -> int:
    return int(sum(counts))                               # ✅ make the declared type true


@dsl.pipeline(name="fan-in-wrong")
def fan_in_wrong() -> int:
    with dsl.ParallelFor([0, 1, 2]) as shard:
        rows = rows_in_shard(shard=shard)
    return total_rows_wrong(counts=dsl.Collected(rows.output)).output


@dsl.pipeline(name="fan-in-right")
def fan_in_right() -> int:
    with dsl.ParallelFor([0, 1, 2]) as shard:
        rows = rows_in_shard(shard=shard)
    return total_rows_right(counts=dsl.Collected(rows.output)).output


_, _, fan_log = run_local(fan_in_wrong)
print("❌", [line.strip() for line in fan_log.splitlines() if "ValueError" in line][-1])
print("✅ total rows:", run_local(fan_in_right)[0].output)

❌ ValueError: Function `total_rows_wrong` returned value of type <class 'float'>; want type <class 'int'>
✅ total rows: 3003


### ❌ Pitfall 4 — Caching a step that isn't a pure function of its inputs

A "snapshot the latest data" step has the same inputs every day, so the cache happily returns yesterday's snapshot.

In [29]:
@dsl.component(**LOCAL)
def snapshot_version(table: str) -> str:
    import datetime
    return f"{table}@{datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S.%f')}"


@dsl.pipeline(name="snapshot-cached")
def snapshot_cached() -> str:
    return snapshot_version(table="warehouse.orders").output


@dsl.pipeline(name="snapshot-fresh")
def snapshot_fresh() -> str:
    task = snapshot_version(table="warehouse.orders")
    task.set_caching_options(enable_caching=False)       # ✅ always re-run
    return task.output


cached = [run_local(snapshot_cached)[0].output for _ in range(2)]
fresh = [run_local(snapshot_fresh)[0].output for _ in range(2)]
print("❌ cached runs:", cached, "→ identical:", cached[0] == cached[1])
print("✅ caching off:", fresh, "→ identical:", fresh[0] == fresh[1])

❌ cached runs: ['warehouse.orders@14:50:13.272869', 'warehouse.orders@14:50:13.272869'] → identical: True
✅ caching off: ['warehouse.orders@14:50:13.344169', 'warehouse.orders@14:50:13.397647'] → identical: False


## 🏋️ Practice Exercises

Try each one before opening the solution. Run the cell: ⏳ means not attempted, ✅ means correct.

### 🟢 Exercise 1 — Your own component
Write a component `word_count(text: str) -> int` (use `**LOCAL`), run it locally on `"pipelines run every step in its own container"`, and store the result in `n_words`.

In [30]:
n_words = None  # TODO
check("n_words", n_words, 8, hint="len(text.split()); run with run_local(word_count, text=...)[0].output")

⏳ n_words: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
@dsl.component(**LOCAL)
def word_count(text: str) -> int:
    return len(text.split())


n_words = run_local(word_count, text="pipelines run every step in its own container")[0].output
check("n_words", n_words, 8)
```
</details>

### 🟢 Exercise 2 — CPU quantities
Implement `cpu_millicores(quantity)` that converts Kubernetes CPU quantities to integer millicores: `"250m"` → 250, `"1.5"` → 1500, `"2"` → 2000.

In [31]:
def cpu_millicores(quantity):
    return None  # TODO


got_millicores = None if cpu_millicores("1") is None else [cpu_millicores(q) for q in ("250m", "1.5", "2", "100m")]
check("cpu_millicores", got_millicores, [250, 1500, 2000, 100], hint="Strip a trailing 'm'; otherwise multiply cores by 1000.")

⏳ cpu_millicores: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def cpu_millicores(quantity):
    quantity = str(quantity)
    return int(quantity[:-1]) if quantity.endswith("m") else round(float(quantity) * 1000)


check("cpu_millicores", [cpu_millicores(q) for q in ("250m", "1.5", "2", "100m")], [250, 1500, 2000, 100])
```
</details>

### 🟢 Exercise 3 — Read a metric back from a run
From the section 5 run folder `train_run_dir`, read the **accuracy** that `evaluate-model` logged into its `metrics` artifact and store it in `logged_accuracy`. The checker recomputes accuracy from the saved model and test set.

In [32]:
logged_accuracy = None  # TODO: use artifact_metadata(...)
from sklearn.metrics import accuracy_score

_test = pd.read_csv(train_run_dir / "load-breast-cancer-data" / "test_set")
_expected_accuracy = round(float(accuracy_score(_test["target"], joblib.load(train_run_dir / "train-logreg" / "model").predict(_test.drop(columns="target")))), 4)
check("logged_accuracy", logged_accuracy, _expected_accuracy, hint="artifact_metadata(train_run_dir, 'evaluate-model', 'metrics')['accuracy']")

⏳ logged_accuracy: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
logged_accuracy = artifact_metadata(train_run_dir, "evaluate-model", "metrics")["accuracy"]
from sklearn.metrics import accuracy_score

_test = pd.read_csv(train_run_dir / "load-breast-cancer-data" / "test_set")
_expected_accuracy = round(float(accuracy_score(_test["target"], joblib.load(train_run_dir / "train-logreg" / "model").predict(_test.drop(columns="target")))), 4)
check("logged_accuracy", logged_accuracy, _expected_accuracy)
```
</details>

### 🟡 Exercise 4 — A branching pipeline
Build a pipeline `parity_pipeline(n: int) -> str` with a component `parity(n: int) -> str` returning `"even"`/`"odd"`, a `dsl.If(... == "even")` branch that calls `shout(message="EVEN")`, a `dsl.Else()` branch that calls `shout(message="ODD")`, and return `dsl.OneOf(...)`. Run it with `n=7` and store the output in `parity_result`.

In [33]:
parity_result = None  # TODO
check("parity_result", parity_result, "ODD", hint="shout can simply return its message; use dsl.OneOf(a.output, b.output).")

⏳ parity_result: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
@dsl.component(**LOCAL)
def parity(n: int) -> str:
    return "even" if n % 2 == 0 else "odd"


@dsl.component(**LOCAL)
def shout(message: str) -> str:
    return message


@dsl.pipeline(name="parity")
def parity_pipeline(n: int) -> str:
    p = parity(n=n)
    with dsl.If(p.output == "even"):
        even = shout(message="EVEN")
    with dsl.Else():
        odd = shout(message="ODD")
    return dsl.OneOf(even.output, odd.output)


parity_result = run_local(parity_pipeline, n=7)[0].output
check("parity_result", parity_result, "ODD")
```
</details>

### 🟡 Exercise 5 — Read the IR
Using `reconstruct(ir)` on the compiled `tune_and_gate` spec, count how many **executable tasks** (not groups) it contains, and store it in `n_executable_tasks`. Cross-check it against `ir["deploymentSpec"]["executors"]`.

In [34]:
n_executable_tasks = None  # TODO
check("n_executable_tasks", n_executable_tasks, len(ir["deploymentSpec"]["executors"]),
      hint="sum(1 for info in reconstruct(ir)[0].values() if info['kind'] == 'task')")

⏳ n_executable_tasks: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
n_executable_tasks = sum(1 for info in reconstruct(ir)[0].values() if info["kind"] == "task")
check("n_executable_tasks", n_executable_tasks, len(ir["deploymentSpec"]["executors"]))
```

Each executable task here has its own executor. (Two tasks using the *same* component with identical settings can share one, so in general the counts are not guaranteed to match.)
</details>

### 🔴 Exercise 6 — Pick the best Katib trial from raw logs (interview-style)
Implement `best_trial(trial_logs, metric, strategy)` for a **maximize** objective. For each trial, extract every `metric=value` pair with `KATIB_DEFAULT_METRICS_FORMAT`; reduce them with the metric strategy — `"max"` (Katib's default for a maximize objective) or `"latest"`; ignore trials that never reported the metric; return `(trial_name, value)` of the best trial.

In [35]:
TRIAL_LOGS = {
    "trial-a": "loading data\nepoch=1 roc_auc=0.941\nepoch=2 roc_auc=0.962\nepoch=3 roc_auc=0.955\n",
    "trial-b": "epoch=1 roc_auc=0.950\nepoch=2 roc_auc=0.958\n",
    "trial-c": '{"epoch": 1, "roc_auc": 0.99}\n',            # JSON on stdout: invisible to the StdOut collector
}


def best_trial(trial_logs, metric, strategy):
    return None  # TODO


got_best = None if best_trial(TRIAL_LOGS, "roc_auc", "max") is None else (
    best_trial(TRIAL_LOGS, "roc_auc", "max"), best_trial(TRIAL_LOGS, "roc_auc", "latest"))
check("best_trial", got_best, (("trial-a", 0.962), ("trial-b", 0.958)),
      hint="re.findall(KATIB_DEFAULT_METRICS_FORMAT, line) yields (name, value, ...) tuples.")

⏳ best_trial: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
TRIAL_LOGS = {
    "trial-a": "loading data\nepoch=1 roc_auc=0.941\nepoch=2 roc_auc=0.962\nepoch=3 roc_auc=0.955\n",
    "trial-b": "epoch=1 roc_auc=0.950\nepoch=2 roc_auc=0.958\n",
    "trial-c": '{"epoch": 1, "roc_auc": 0.99}\n',
}


def best_trial(trial_logs, metric, strategy):
    reduce = {"max": max, "latest": lambda values: values[-1]}[strategy]
    scores = {}
    for trial, log in trial_logs.items():
        values = [float(value) for line in log.splitlines()
                  for name, value, *_ in re.findall(KATIB_DEFAULT_METRICS_FORMAT, line) if name == metric]
        if values:
            scores[trial] = reduce(values)
    winner = max(scores, key=scores.get)
    return winner, scores[winner]


check("best_trial", (best_trial(TRIAL_LOGS, "roc_auc", "max"), best_trial(TRIAL_LOGS, "roc_auc", "latest")),
      (("trial-a", 0.962), ("trial-b", 0.958)))
```

The strategy changes the winner: trial-a peaked higher but ended lower. And trial-c's excellent-looking JSON result is invisible to the default collector — a real bug that silently discards your best run.
</details>

## 🚀 Mini Project: Train → Evaluate → Conditional Register

**Goal:** a real KFP v2 pipeline on the **breast cancer diagnostic dataset** (569 patients, 30 features from digitized biopsy images) that only registers a model when it is good enough — and proves the gate works by running twice with different thresholds.

```
  load-breast-cancer-data ─┬─► ParallelFor C ∈ {0.01, 0.1, 1, 10}: cross-validate-logreg (train split only)
                           │         └──► pick-best (C, cv_auc)
                           └─► train-logreg (best C, full train split) ─► evaluate-model (test split, touched once)
                                                                             │
                                 dsl.If(test_auc ≥ min_test_auc) ─► register-model (copies model + metadata)
                                 dsl.Else()                       ─► reject-model
```

**No leakage:** `C` is chosen by 5-fold cross-validation on the training split; the test split is used exactly once, by `evaluate-model`, and that number drives the gate.

### Step 1 — Components for the gate

In [36]:
@dsl.component(**LOCAL)
def register_model(model: Input[Model], test_auc: float, min_auc: float, best_C: float,
                   registered_model: Output[Model]) -> str:
    import shutil
    shutil.copyfile(model.path, registered_model.path)          # in production: push to a model registry
    registered_model.metadata.update({"stage": "candidate", "test_auc": round(test_auc, 4), "C": best_C,
                                      "gate": f"test_auc >= {min_auc}"})
    return f"registered: test AUC {test_auc:.4f} >= {min_auc}"


@dsl.component(**LOCAL)
def reject_model(test_auc: float, min_auc: float) -> str:
    return f"rejected: test AUC {test_auc:.4f} < {min_auc}"

### Step 2 — The pipeline

In [37]:
@dsl.pipeline(name="train-evaluate-register", description="Tune C on train folds, evaluate once on test, register only if good enough")
def train_evaluate_register(min_test_auc: float = 0.97) -> str:
    data = load_breast_cancer_data(test_size=0.25, seed=0)
    with dsl.ParallelFor(items=[0.01, 0.1, 1.0, 10.0], parallelism=2) as C:
        cv = cross_validate_logreg(train_set=data.outputs["train_set"], C=C)
    best = pick_best(results=dsl.Collected(cv.output))
    final = train_logreg(train_set=data.outputs["train_set"], C=best.outputs["C"])
    final.set_cpu_request("1").set_memory_request("1Gi")                    # compiled for the cluster
    scored = evaluate_model(test_set=data.outputs["test_set"], model=final.outputs["model"])
    with dsl.If(scored.outputs["Output"] >= min_test_auc, name="good-enough"):
        registered = register_model(model=final.outputs["model"], test_auc=scored.outputs["Output"],
                                    min_auc=min_test_auc, best_C=best.outputs["C"])
    with dsl.Else(name="not-good-enough"):
        rejected = reject_model(test_auc=scored.outputs["Output"], min_auc=min_test_auc)
    return dsl.OneOf(registered.outputs["Output"], rejected.output)


compiler.Compiler().compile(train_evaluate_register, str(SPECS / "train_evaluate_register.yaml"))
mp_nodes, mp_control, mp_data = reconstruct(yaml.safe_load((SPECS / "train_evaluate_register.yaml").read_text()))
print("executable tasks:", sorted(n for n, info in mp_nodes.items() if info["kind"] == "task"))
print("data edges into the gate branches:", sorted(e for e in mp_data if e[2] in ("register-model", "reject-model")))

executable tasks: ['cross-validate-logreg', 'evaluate-model', 'load-breast-cancer-data', 'pick-best', 'register-model', 'reject-model', 'train-logreg']
data edges into the gate branches: [('evaluate-model', 'Output', 'register-model', 'test_auc'), ('evaluate-model', 'Output', 'reject-model', 'test_auc'), ('pick-best', 'C', 'register-model', 'best_C'), ('train-logreg', 'model', 'register-model', 'model')]


### Step 3 — Run 1: a realistic threshold (test AUC ≥ 0.97)

Sections 5–7 already ran several of these exact steps, so a cached run would skip them and leave no fresh files to inspect. Run 1 therefore executes with caching **off**; run 2 turns it back on.

In [38]:
init_local(enable_caching=False)
try:
    run1, seconds1, log1 = run_local(train_evaluate_register, min_test_auc=0.97)
finally:
    init_local(enable_caching=True)
run1_dir = latest_run_dir("train-evaluate-register")
statuses1 = dict(task_statuses(log1))
print(f"output: {run1.output!r} ({seconds1:.1f} s)")
print("statuses:", statuses1)

best1 = executor_output(run1_dir, "pick-best")["parameterValues"]
test_auc1 = executor_output(run1_dir, "evaluate-model")["parameterValues"]["Output"]
cv_results = [executor_output(d, "cross-validate-logreg")["parameterValues"]["Output"]
              for d in sorted(run1_dir.glob("for-loop-*-iteration-*"))]
print("cross-validation results (train folds only):", cv_results)
print(f"best C = {best1['C']} (CV AUC {best1['cv_auc']:.4f}) → test AUC {test_auc1:.4f}")

ℹ️ local runner: Task 'train-logreg': settings ['cpu_request', 'memory_request'] have no effect in the current local runner and will be ignored.
output: 'registered: test AUC 0.9929 >= 0.97' (6.0 s)
statuses: {'load-breast-cancer-data': 'SUCCESS', 'cross-validate-logreg': 'SUCCESS', 'pick-best': 'SUCCESS', 'train-logreg': 'SUCCESS', 'evaluate-model': 'SUCCESS', 'register-model': 'SUCCESS'}
cross-validation results (train folds only): [{'C': 0.01, 'cv_auc': 0.99422}, {'C': 0.1, 'cv_auc': 0.99587}, {'C': 1.0, 'cv_auc': 0.9954}, {'C': 10.0, 'cv_auc': 0.98546}]
best C = 0.1 (CV AUC 0.9959) → test AUC 0.9929


### Step 4 — Read back the registered artifact and verify it

In [39]:
from sklearn.metrics import roc_auc_score

registered_info = executor_output(run1_dir, "register-model")["artifacts"]["registered_model"]["artifacts"][0]
registered_path = run1_dir / "register-model" / "registered_model"
print("registered artifact:", rel(registered_path), f"({registered_path.stat().st_size:,} bytes)")
print("metadata:", registered_info["metadata"])

reloaded = joblib.load(registered_path)
test_df = pd.read_csv(run1_dir / "load-breast-cancer-data" / "test_set")
recomputed_auc = roc_auc_score(test_df["target"], reloaded.predict_proba(test_df.drop(columns="target"))[:, 1])
assert math.isclose(recomputed_auc, test_auc1, rel_tol=1e-9)
assert reloaded.get_params()["logisticregression__C"] == best1["C"]
print(f"✅ reloaded model reproduces the gate's test AUC ({recomputed_auc:.4f}) and uses the tuned C={best1['C']}")

registered artifact: $PIPELINE_ROOT/train-evaluate-register-2026-09-15-10-50-13-531899/register-model/registered_model (3,105 bytes)
metadata: {'stage': 'candidate', 'test_auc': 0.9929, 'C': 0.1, 'gate': 'test_auc >= 0.97'}
✅ reloaded model reproduces the gate's test AUC (0.9929) and uses the tuned C=0.1


### Step 5 — Run 2: an unreachable threshold (test AUC ≥ 0.999)

Same data and code, so steps whose component and inputs are unchanged are served from the **cache**; the gate now takes the other branch.

In [40]:
run2, seconds2, log2 = run_local(train_evaluate_register, min_test_auc=0.999)
statuses2 = dict(task_statuses(log2))
print(f"output: {run2.output!r} ({seconds2:.1f} s)")
print("cache hits:", cache_hits(log2))
print("statuses:", statuses2)

ℹ️ local runner: Task 'train-logreg': settings ['cpu_request', 'memory_request'] have no effect in the current local runner and will be ignored.
output: 'rejected: test AUC 0.9929 < 0.999' (5.1 s)
cache hits: ['pick-best']
statuses: {'load-breast-cancer-data': 'SUCCESS', 'cross-validate-logreg': 'SUCCESS', 'train-logreg': 'SUCCESS', 'evaluate-model': 'SUCCESS', 'reject-model': 'SUCCESS'}


### Step 6 — Conclusions (computed)

In [41]:
def branch_that_ran(statuses):
    ran = [name for name in ("register-model", "reject-model") if statuses.get(name) == "SUCCESS"]
    assert len(ran) == 1, statuses
    return ran[0]


for label, threshold, statuses, output in (("run 1", 0.97, statuses1, run1.output), ("run 2", 0.999, statuses2, run2.output)):
    expected = "register-model" if test_auc1 >= threshold else "reject-model"
    assert branch_that_ran(statuses) == expected, (label, statuses)
    print(f"{label}: test AUC {test_auc1:.4f} vs threshold {threshold} → {branch_that_ran(statuses)} ran → {output!r}")

cv_spread = max(r["cv_auc"] for r in cv_results) - min(r["cv_auc"] for r in cv_results)
print(f"\nTuning: CV AUC varied by {cv_spread:.4f} across the C values; C={best1['C']} was best.")
print(f"Generalization: CV AUC {best1['cv_auc']:.4f} vs untouched test AUC {test_auc1:.4f} "
      f"({'within' if abs(best1['cv_auc'] - test_auc1) < 0.02 else 'more than'} 0.02 of each other).")
reused = cache_hits(log2)
print(f"Caching: run 2 re-used {len(reused)} step{'s' if len(reused) != 1 else ''} {reused} from earlier cached runs; "
      f"the other steps had no matching cache entry (run 1 ran with caching off, so it wrote none) and executed ({seconds2:.1f} s).")

run 1: test AUC 0.9929 vs threshold 0.97 → register-model ran → 'registered: test AUC 0.9929 >= 0.97'
run 2: test AUC 0.9929 vs threshold 0.999 → reject-model ran → 'rejected: test AUC 0.9929 < 0.999'

Tuning: CV AUC varied by 0.0104 across the C values; C=0.1 was best.
Generalization: CV AUC 0.9959 vs untouched test AUC 0.9929 (within 0.02 of each other).
Caching: run 2 re-used 1 step ['pick-best'] from earlier cached runs; the other steps had no matching cache entry (run 1 ran with caching off, so it wrote none) and executed (5.1 s).


**Stretch goals**

1. Replace the `Else` branch with `Elif(test_auc >= 0.9)` → a `request_human_review` step, and add an `ExitHandler` that posts the run's final status.
2. Make `register-model` push to a real registry (e.g. MLflow with an alias, see [MLflow](01_MLflow.ipynb)) and pass the registered version as a parameter output.
3. Compare the challenger with the **current champion** on the same test set before registering (like the Airflow project) instead of a fixed threshold.
4. Write the Katib `Experiment` from section 9 for this dataset and use its best `C` as a pipeline parameter.
5. Turn on Docker (and `pip install docker` in this environment) and re-run with `kfp.local.DockerRunner()` using a custom image with scikit-learn preinstalled.

### 🗣️ How to talk about this in an interview
- "I built a KFP v2 pipeline with lightweight Python components and typed artifacts: data as `Dataset`s, the model as a `Model`, metrics as `Metrics`/`ClassificationMetrics`, and the test AUC as a parameter so it can drive control flow."
- "Hyperparameters were tuned with a `ParallelFor` fan-out of cross-validation on the training split and a `Collected` fan-in; the test split was touched once, by the evaluation step that feeds the `dsl.If` registration gate."
- "I proved the gate by running it twice with different thresholds — one registered, one rejected — and verified the registered artifact by reloading it and reproducing the gate's AUC."
- "Caching reuses a step only when its component definition and inputs match a stored entry, so I ran the first run uncached to get freshly written artifacts to inspect, and I disable caching on steps that read external, changing data."
- "The compiled YAML is what runs on Kubeflow Pipelines or Vertex AI; locally I used `kfp.local` with the subprocess runner, and on a cluster I'd use a pinned image instead of `packages_to_install`."

## 🎤 Interview Q&A

Try answering **out loud** before opening each answer.

### 🧠 Concepts

**Q1. What are components, tasks and pipelines in Kubeflow Pipelines v2?**

<details><summary>Show answer</summary>

- **30-second answer:** A component is a reusable step with a typed interface (inputs/outputs) and an implementation — usually a container command; a task is one use of a component inside a pipeline with concrete arguments; a pipeline is a DAG of tasks (and control-flow groups) that compiles into one pipeline spec.
- **Go deeper:** Lightweight Python components (`@dsl.component`) ship the function source and run it in `base_image`; container components (`@dsl.container_component`) run an arbitrary image/command; pipelines can be nested as components. Each task runs as its own pod on a cluster.
- **❌ Common wrong answer:** "A pipeline is a Python script that runs top to bottom on one machine." The Python only *defines* the graph; each step runs separately.

</details>

**Q2. Parameters vs artifacts — what's the difference and when do you use each?**

<details><summary>Show answer</summary>

- **30-second answer:** Parameters are small typed values (numbers, strings, lists, dicts) passed by value; artifacts are files or directories (Dataset, Model, Metrics…) stored in object storage under the pipeline root, passed by reference with metadata and lineage.
- **Go deeper:** Only parameters can be used in `dsl.If` conditions or as `ParallelFor` items; `Metrics`/`ClassificationMetrics` are artifacts whose content is metadata the UI visualizes. A common pattern returns a metric both ways: logged in `Metrics` for tracking, returned as a `float` for gating.
- **❌ Common wrong answer:** "Just pass the DataFrame as a JSON string parameter." That bloats run metadata and loses lineage.

</details>

**Q3. Explain Kubernetes resource requests and limits. What happens when a training pod exceeds them?**

<details><summary>Show answer</summary>

- **30-second answer:** Requests are what the scheduler reserves when placing the pod; limits are hard caps. Exceeding the CPU limit throttles the container; exceeding the memory limit kills it (`OOMKilled`, exit code 137). If no node can satisfy the requests, the pod stays `Pending`.
- **Go deeper:** Pods with requests equal to limits for every container get the `Guaranteed` QoS class and are evicted last under node pressure. GPUs are requested as extended resources (`nvidia.com/gpu`) and can't be overcommitted. In KFP: `.set_cpu_request()`, `.set_memory_limit()`, `.set_accelerator_type()/.set_accelerator_limit()`.
- **❌ Common wrong answer:** "Limits are reserved for the pod" or "exceeding memory just slows it down."

</details>

**Q4. How does KFP decide to reuse a cached step, and when should you turn caching off?**

<details><summary>Show answer</summary>

- **30-second answer:** The cache key combines the component's definition (image, command) with its input values and input artifacts; if a previous successful execution has the same key, its outputs are reused. Turn it off for steps that aren't pure functions of their inputs: reading "the latest" data, calling external services, or randomness not passed in as an input.
- **Go deeper:** Per task: `task.set_caching_options(enable_caching=False)`; per run: `enable_caching=` when submitting. Locally (`kfp.local`) caching is opt-in via `local.init(enable_caching=True)` and in this notebook a step with metadata-only `Metrics` outputs was re-executed (section 7). A changed upstream output changes downstream keys, so only affected steps re-run.
- **❌ Common wrong answer:** "Caching compares the step's name" — renaming nothing and changing an input still re-runs; changing *data behind a fixed path* does **not** change the key, which is how stale results happen.

</details>

**Q5. What does the KFP compiler produce, and how does it relate to Vertex AI Pipelines?**

<details><summary>Show answer</summary>

- **30-second answer:** A pipeline spec (IR) YAML: the root DAG, component interfaces, and `deploymentSpec` executors (image, command, resources). Kubeflow Pipelines and Google Cloud's managed pipelines service both execute this spec, so one pipeline definition runs on either.
- **Go deeper:** The spec lists `dependentTasks`, input wiring (`taskOutputArtifact`, `componentInputParameter`), `triggerPolicy` conditions and `iteratorPolicy` loops — section 8 and Build It From Scratch parsed exactly these. Version the YAML in git and review diffs.
- **❌ Common wrong answer:** "Vertex AI needs its own pipeline SDK, so KFP code must be rewritten."

</details>

**Q6. How does Katib read a trial's metric? Your experiment shows "metrics not available" — why?**

<details><summary>Show answer</summary>

- **30-second answer:** The default collector is `StdOut`: it parses the training container's stdout with the regex `([\w|-]+)\s*=\s*([+-]?\d*(\.\d+)?([Ee][+-]?\d+)?)`, i.e. `name=value` lines, and looks for `objectiveMetricName`. Printing JSON, using a different metric name, or writing to a file without configuring the `File` collector means Katib finds nothing.
- **Go deeper:** For JSON, use `metricsCollectorSpec.collector.kind: File` with `source.fileSystemPath` (`path`, `kind: File`, `format: JSON`); custom formats use `source.filter.metricsFormat`. With `metricStrategies`, each metric is reduced by `max`, `min` or `latest`; the default equals the objective type (maximize → `max`). Exercise 6 implemented this.
- **❌ Common wrong answer:** "Katib reads the value the training function returns."

</details>

**Q7. What is a KServe `InferenceService`, and what does a v2 inference request look like?**

<details><summary>Show answer</summary>

- **30-second answer:** A Kubernetes resource that deploys a model server: `spec.predictor.model.modelFormat` picks a runtime (e.g. `sklearn` → `kserve-sklearnserver`), `storageUri` points to the model directory, plus replicas and resources. With `protocolVersion: v2` clients `POST /v2/models/<name>/infer` with `{"inputs": [{"name", "shape", "datatype", "data"}]}` and get `{"outputs": [...]}`.
- **Go deeper:** KServe adds autoscaling (including scale-to-zero in serverless mode), canary rollouts via `canaryTrafficPercent`, transformers/explainers, and multi-model serving. Section 10 built a real v2 payload and emulated the server's decode → predict → encode.
- **❌ Common wrong answer:** "`storageUri` must point to the `model.joblib` file itself" — it points to the directory.

</details>

### 💻 Coding

**Q8. Write a KFP pipeline that deploys a model only if its test AUC is at least 0.95.**

<details><summary>Show answer</summary>

- **30-second answer:**
  ```python
  @dsl.pipeline(name="gated-deploy")
  def gated_deploy(min_auc: float = 0.95):
      data = load_data()
      model = train(train_set=data.outputs["train_set"])
      auc = evaluate(test_set=data.outputs["test_set"], model=model.outputs["model"]).outputs["auc"]
      with dsl.If(auc >= min_auc, name="deploy"):
          deploy(model=model.outputs["model"])
      with dsl.Else(name="skip"):
          notify(message="model below threshold")
  ```
- **Go deeper:** `evaluate` returns the AUC as a **parameter** (e.g. a `NamedTuple` field) because conditions can't read artifacts; wrap deploy/notify in an `ExitHandler` if a failure must also alert. The mini project runs exactly this pattern twice.
- **❌ Common wrong answer:** Using a Python `if auc >= min_auc:` inside the pipeline function — at definition time `auc` is a placeholder channel, not a number.

</details>

**Q9. How do you fan out training over a list of hyperparameters and pick the best in KFP?**

<details><summary>Show answer</summary>

- **30-second answer:** `with dsl.ParallelFor(items=[...], parallelism=k) as value: score = cv_score(C=value)`, then `best = pick_best(results=dsl.Collected(score.output))`, and use `best.outputs[...]` downstream.
- **Go deeper:** Items can be a pipeline parameter or another task's list output; `parallelism` limits concurrent pods; tune on validation/CV, not the test set. For large searches, use Katib (Bayesian optimization, early stopping) instead of a hand-rolled grid.
- **❌ Common wrong answer:** A Python `for` loop creating tasks — it works for a *static* list, but you lose the loop grouping and parallelism controls, and it can't iterate over a runtime value.

</details>

### 🐛 Debugging Scenarios

**Q10. A component works in your notebook but fails on the cluster with `NameError` or `ModuleNotFoundError`. What's wrong?**

<details><summary>Show answer</summary>

- **30-second answer:** Lightweight components ship only the function body. Helpers or imports defined outside the function don't exist in the container (`NameError`), and packages missing from `base_image`/`packages_to_install` aren't installed (`ModuleNotFoundError`). Move imports and helpers inside the function and declare dependencies or use a proper image.
- **Go deeper:** Pitfall 1 reproduced the `NameError`. For shared code across components, build a container image with your package installed, or use `additional_funcs`/containerized Python components. Pin versions so the cluster matches your tests.
- **❌ Common wrong answer:** "Restart the pipeline" or "it's a Kubernetes networking issue."

</details>

**Q11. A training step is stuck in `Pending`, or restarts with `OOMKilled`. How do you debug it?**

<details><summary>Show answer</summary>

- **30-second answer:** `kubectl describe pod <pod>`: `Pending` with "Insufficient cpu/memory/nvidia.com/gpu" means requests don't fit any node (lower requests, add a node pool, check namespace quotas); `OOMKilled` / exit code 137 means the memory limit is too low for the job (raise the limit, reduce batch size, stream data).
- **Go deeper:** Also check node selectors/tolerations for GPU pools, PVC binding, and image pull errors (`ImagePullBackOff`). Use metrics (container memory working set) to right-size requests.
- **❌ Common wrong answer:** "Increase retries" — the same pod spec will fail the same way.

</details>

**Q12. A nightly pipeline keeps producing the same model even though new data arrives. Why?**

<details><summary>Show answer</summary>

- **30-second answer:** The data-loading step reads "latest" data from a fixed path or query, so its inputs (and cache key) never change and KFP serves the cached output. Disable caching on that step, or pass a changing input such as the data snapshot date/version so the key changes when the data does.
- **Go deeper:** Pitfall 4 showed a cached "snapshot" returning an identical timestamp. Passing explicit data versions (e.g. a DVC revision or partition date) also improves lineage and reproducibility.
- **❌ Common wrong answer:** "Delete the cache folder every night."

</details>

### 🏗️ Design

**Q13. Design a retraining-and-deployment platform on Kubernetes for a team of 20 data scientists.**

<details><summary>Show answer</summary>

- **30-second answer:** KFP (or a managed equivalent) for pipelines with pinned component images, artifacts in object storage and per-team namespaces with quotas; Katib for tuning; a model registry for versions and approval; KServe `InferenceService`s with canary rollouts; monitoring for drift and latency that can trigger retraining; CI that compiles pipelines and runs them locally before submission.
- **Go deeper:** Scheduling and cross-system dependencies can live in Airflow (trigger the KFP run when data lands); GPU node pools with autoscaling; secrets via service accounts/workload identity; reproducibility through data versions + image digests + compiled specs in git.
- **❌ Common wrong answer:** "One big notebook on a GPU VM that someone runs manually."

</details>

**Q14. Kubeflow Pipelines or Airflow for an ML workflow — how do you decide?**

<details><summary>Show answer</summary>

- **30-second answer:** KFP when steps are ML workloads needing per-step containers, GPUs, typed artifacts, lineage and caching on Kubernetes (or Vertex AI); Airflow when the workflow is part of broader data orchestration with many integrations, rich schedules and backfills. Many teams use Airflow to trigger KFP runs.
- **Go deeper:** Consider existing infrastructure (is there a Kubernetes cluster?), who maintains it, local testability (`kfp.local`, `dag.test()`), and data passing (artifacts vs XCom references).
- **❌ Common wrong answer:** "KFP is always better for ML" — without Kubernetes expertise the operational cost can outweigh the benefits.

</details>

## 🧪 Quick Quiz

Predict the answer, then reveal. Each one was checked by running code in this notebook.

**1.** `@dsl.component def add(a: int, b: int) -> int` — what is the name of its single output in `add.component_spec.outputs`?
<details><summary>Answer</summary>

`'Output'` (section 3 printed `{'Output': 'Integer'}`), which is why `task.outputs["Output"]` works when a component also has artifact outputs.
</details>

**2.** A task comes from a component with a `float` return value **and** an `Output[Metrics]` argument. What happens if the pipeline uses `task.output`?
<details><summary>Answer</summary>

`AttributeError: The task has multiple outputs. Please reference the output by its name.` — raised while *defining* the pipeline (Pitfall 2).
</details>

**3.** How many bytes are `memory: 1Gi` and `memory: 1G`?
<details><summary>Answer</summary>

`1Gi` = 1,073,741,824 bytes (2³⁰); `1G` = 1,000,000,000 bytes (10⁹). Section 1's `memory_bytes()` computed the 4 GiB vs 4 GB difference.
</details>

**4.** A Katib trial prints `{"roc_auc": 0.97}` and the experiment uses the default metrics collector. What value does Katib record?
<details><summary>Answer</summary>

None — the default `StdOut` collector only matches `name=value` lines, so the metric is unavailable (section 9 and Exercise 6's `trial-c`).
</details>

**5.** You re-run `train_pipeline(C=1.0)` locally with caching enabled. Which steps are skipped?
<details><summary>Answer</summary>

In this notebook's run, `load-breast-cancer-data` and `train-logreg` were cache hits and `evaluate-model` executed again — its `Metrics`/`ClassificationMetrics` outputs are metadata-only artifacts (section 7). Changing `C` also re-runs `train-logreg`.
</details>

## 📚 Resources

### 📖 Official Docs
- [Kubeflow Pipelines | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/) — overview and concepts
- [Execute KFP pipelines locally | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/execute-kfp-pipelines-locally/) — `kfp.local`, SubprocessRunner and DockerRunner
- [Lightweight Python Components | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/user-guides/components/lightweight-python-components/) · [Create, use, pass, and track ML artifacts | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/user-guides/data-handling/artifacts/)
- [Control Flow | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/control-flow/) · [Use Caching | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/caching/) · [Compile a Pipeline | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/compile-a-pipeline/)
- [KFP SDK API Reference](https://kubeflow-pipelines.readthedocs.io/en/stable/) — every `dsl`, `compiler`, `local` and `Client` API
- [Kubeflow Katib | Kubeflow](https://www.kubeflow.org/docs/components/katib/) · [How to Configure Experiment | Kubeflow](https://www.kubeflow.org/docs/components/katib/user-guides/hp-tuning/configure-experiment/) · [How to Configure Metrics Collector | Kubeflow](https://www.kubeflow.org/docs/components/katib/user-guides/metrics-collector/)
- [KServe](https://kserve.github.io/website/) · [Scikit-learn | KServe](https://kserve.github.io/website/docs/model-serving/predictive-inference/frameworks/sklearn) — `InferenceService` and the v2 protocol
- [Kubeflow Trainer | Kubeflow](https://www.kubeflow.org/docs/components/trainer/) — distributed training on Kubernetes
- Kubernetes: [Pods | Kubernetes](https://kubernetes.io/docs/concepts/workloads/pods/) · [Resource Management for Pods and Containers | Kubernetes](https://kubernetes.io/docs/concepts/configuration/manage-resources-containers/)

### 🎥 Videos
- [Google Cloud Tech — How to build a Kubeflow Pipeline](https://www.youtube.com/watch?v=JY7za08sAIU) (~5 min) — the component → pipeline → run workflow in one short visual walkthrough
- [Kubeflow — Kubeflow Pipelines 2.0: Introduction & Roadmap (Kubeflow Summit 2022)](https://www.youtube.com/watch?v=JiM69LyUvEM) (~50 min) — the maintainers explain why v2 introduced the IR spec, artifacts and the new SDK
- [Abhishek.Veeramalla — KubeFlow Pipelines Zero to Hero with a Realtime MLOps Project](https://www.youtube.com/watch?v=5iOQcGfcZe4) (~1 h) — installs Kubeflow on a real cluster, the part this notebook can't do on a laptop

### 📄 Papers
- [George et al. (2020) — A Scalable and Cloud-Native Hyperparameter Tuning System](https://arxiv.org/abs/2006.02085) — the Katib design paper
- [Burns et al. (2016) — Borg, Omega, and Kubernetes](https://research.google/pubs/borg-omega-and-kubernetes/) — why Kubernetes works the way it does, from its creators
- [Baylor et al. (2017) — TFX: A TensorFlow-Based Production-Scale Machine Learning Platform](https://research.google/pubs/tfx-a-tensorflow-based-production-scale-machine-learning-platform/) — the pipeline + validation-gate design KFP-style platforms follow

### 📘 Books & Courses
- [Learn Kubernetes Basics (official interactive tutorial)](https://kubernetes.io/docs/tutorials/kubernetes-basics/) — deploy, scale and update an app in the browser
- [Run a Pipeline | Kubeflow](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/run-a-pipeline/) — the official guide to submitting compiled pipelines to a cluster once you have one

### 🏋️ Practice
- [Kubeflow Pipelines samples (GitHub)](https://github.com/kubeflow/pipelines/tree/master/samples) — official pipelines to read and run locally with `kfp.local`
- [Katib examples (GitHub)](https://github.com/kubeflow/katib/tree/master/examples/v1beta1) — real `Experiment` YAMLs for every algorithm and metrics collector

## 📝 Summary Cheat Sheet

| Concept | What it does | Key API / rule |
|---|---|---|
| Kubernetes basics | where ML workloads run | image → container → pod on a node, in a namespace; requests schedule, limits cap (memory → OOMKilled) |
| Kubeflow ecosystem | ML tools on K8s | Pipelines · Katib · KServe · Trainer · Notebooks; Google Cloud runs the KFP spec as a managed service |
| Component | one typed step | `@dsl.component(base_image=..., packages_to_install=[...])`; imports inside |
| Parameters vs artifacts | small values vs files + metadata | `-> float`; `Input[Dataset]`, `Output[Model]`, `.path`, `.metadata`, `Metrics.log_metric` |
| Pipeline | wire tasks into a DAG | `@dsl.pipeline`; `task.output` / `task.outputs["name"]`; `.set_cpu_request()`, `.set_retry()` |
| Local execution | run for real without a cluster | `local.init(runner=local.SubprocessRunner(use_venv=False), pipeline_root=...)` |
| Control flow | branch, loop, always-run | `dsl.If/Elif/Else`, `dsl.OneOf`, `dsl.ParallelFor(..., parallelism=)`, `dsl.Collected`, `dsl.ExitHandler` |
| Caching | skip unchanged steps | key = component + inputs; `task.set_caching_options(enable_caching=False)` for non-deterministic steps |
| Compile & IR | portable spec | `compiler.Compiler().compile(pipeline, "p.yaml")`; `root.dag.tasks`, `components`, `deploymentSpec.executors` |
| Katib | tune on K8s | `Experiment`: objective, algorithm, parameters, trialTemplate; `StdOut` collector parses `name=value` |
| KServe | serve on K8s | `InferenceService`: `modelFormat`, `storageUri` (directory), `protocolVersion: v2` → `/v2/models/<name>/infer` |
| Cluster submission | run on Kubeflow | `kfp.Client(host=...).create_run_from_pipeline_func(pipeline, arguments=...)` |

## ➡️ What's Next

**[Module 10 · FastAPI](../10_Model_Serving/01_FastAPI.ipynb)** — you've seen how platforms like KServe serve models on Kubernetes; next you'll build a model-serving API yourself with FastAPI, the most common way ML engineers put a model behind an HTTP endpoint.